# Airline recovery analysis post COVID and analysis of other performance factors: dashboard (version 5)
Run after the download and cleaning notebooks. Reads `data/clean/` and `data/interim/`, downloads the city-pair file once into `data/raw/github/` if missing, and writes `outputs/dashboard/dgca_dashboard_v5_<date>.html` (self-contained, opens offline).

**Changes in this version (v5)**
- New page title: "Airline recovery analysis post COVID and analysis of other performance factors for several Indian domestic airlines"; the subtitle notes that the scope started as recovery only and then widened.
- The four highlights each sit in their own outlined box.
- The contents list (and "How to read this page") is replaced by a row of square jump buttons, one per part of the page; the page stays one long scroll and the button of the section on screen is highlighted.
- Three charts moved out of the supporting section to where they belong: the brand sunburst to section 5 (after "Is size the same as performance?"), the airline distances and peer groups to the end of section 9.
- The supporting section is renamed "Other plots" and now comes last, after the credits.
- Conclusions are seven numbered findings, every number computed from the data, directly above "Credits and limitations".
- "Credits" is now "Credits and limitations", with sub-headings: Data and sources (Jet Airways source removed; Vonter repository noted as ODbL), Contribution split, LLM Usage, robots.txt and Limitations.
- Every chart and all page text use Arial.

**Changes in version 4**
- The whole page is built from Plotly figures: header and KPI tiles (`go.Indicator`), the metric table (`go.Table`), text, notes and conclusions (text-only figures).
- Every chart has a notes panel: how it works, what it tells us, and observations for the presentation. All numbers in the notes are computed from the data.
- Rank year by year: fixed x-axis (it was squeezed to one side because label positions were read as numbers).
- Each airline vs its pre-COVID level: seat-km added next to passenger-km, plus why only 5 airlines qualify and what explains the gains and drops.
- Two towers and a plane: Play / Pause / Restart buttons, smooth motion, the plane shows the market size as it flies.
- Jet Airways chart removed; it is now a point in the conclusions.
- Filter funnel: each description sits on its own box.
- Busy months: polar chart replaced by a year x month calendar heatmap.
- Route churn recoloured; route histogram lists the routes in the two tails on hover; cities split into three short panels.
- New: five-pair explorer with a drop-down (shows where a correlation across airlines differs from the one inside an airline).
- Section order: the data before and after cleaning comes first; the scorecard sits just before the conclusions; supporting charts (contour, sunburst, pairs, distance, peer groups) are in an Extras section.
- The plane now takes seven steps, one per airline, each moving it by that airline's change in passenger-km; chart titles no longer get cut off; the conclusions are written as prose; a Credits section closes the page.

## 1. Setup and data

In [1]:
# SETUP: paths, libraries and data. This notebook READS data/ (and downloads the city-pair file once into data/raw/github/ if it is missing)
# and WRITES only outputs/dashboard/.
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)
import calendar, json, textwrap, html as _html
from pathlib import Path
import numpy as np, pandas as pd, requests
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from plotly.colors import sample_colorscale
from plotly.offline import get_plotlyjs
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import pdist, squareform
from IPython.display import display, Markdown

SHOW_FIGURES = True        # False = only build the HTML file (faster, smaller notebook)
TODAY = pd.Timestamp.today().date().isoformat()
_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "clean").is_dir())
RAW, INTERIM, CLEAN = (PROJECT_ROOT / "data" / d for d in ("raw", "interim", "clean"))
OUT = PROJECT_ROOT / "outputs" / "dashboard"; OUT.mkdir(parents=True, exist_ok=True)
latest = lambda folder, pat: sorted(folder.glob(pat))[-1]

clean = pd.read_parquet(latest(CLEAN, "dgca_domestic_clean_*.parquet"))                    # domestic, cleaned
interim = pd.read_parquet(latest(INTERIM, "dgca_domestic_airline_monthly_*.parquet"))      # includes the international sections
rec = pd.read_csv(latest(CLEAN, "total_domestic_reconciliation_*.csv"), parse_dates=["Date"])
LAST = rec.loc[rec.within_2pct, "Date"].max()          # last month where the airlines add up to DGCA's own total
MIN_PAX = 25_000            # an airline is scored only if it averaged at least 25,000 passengers a month over the window
MIN_BASE_SHARE = 0.005      # 2019 comparisons need at least 0.5% of 2019 passengers
MATURE = 36                 # months an airline needs before its momentum scores count in full (countermeasure for new airlines)
print(f"clean rows: {len(clean):,} | analysis ends: {LAST:%B %Y}")


clean rows: 1,309 | analysis ends: August 2026


## 2. Monthly matrices and the pre-COVID baseline

In [2]:
# GROUP-MONTH MATRICES (one column per airline group): domestic from the clean table, international from the interim table
months = pd.date_range("2017-01-01", LAST, freq="MS")
FIELDS = dict(pax="Passenger Number", rpk="Passenger Kilometers", ask="Seat Kilometers", dep="Aircraft Number", hrs="Aircraft Hours")
# units: pax = passengers, rpk/ask = THOUSAND passenger-km / seat-km, dep = departures, hrs = aircraft hours
def matrices(df):
    return {k: df.groupby(["Group", "Date"])[c].sum(min_count=1).unstack("Group").reindex(months).astype(float) for k, c in FIELDS.items()}
def zero_fill(Pm, start, end):
    """Months with no report inside an airline's active period are a structural zero (nobody flew), not unknown values."""
    for k in Pm:
        for g in Pm[k].columns:
            s = Pm["pax"][g]; f, l = s.first_valid_index(), s.last_valid_index()
            if f is None: continue
            gap = (Pm[k].index >= max(f, pd.Timestamp(start))) & (Pm[k].index <= min(l, pd.Timestamp(end))) & Pm[k][g].isna()
            Pm[k].loc[gap, g] = 0.0
    return Pm
sd = clean[(clean.Type == "ScheduledDomestic") & (clean.Date <= LAST)]
P = zero_fill(matrices(sd), "2020-04-01", "2020-05-01")        # domestic: nobody flew in April-May 2020
mk, mk_rpk = P["pax"].sum(axis=1), P["rpk"].sum(axis=1)       # whole market
GMAP = dict(clean[["Airline", "Group"]].drop_duplicates().values)
it = interim[(interim.Type == "ScheduledInternational") & ~interim.Airline.isin(["Total Domestic", "Total International"])].copy()
it["Date"] = pd.to_datetime(dict(year=it.Year, month=it.Month, day=1)); it = it[it.Date <= LAST]
it["Group"] = it.Airline.map(GMAP).fillna(it.Airline)
PI = zero_fill(matrices(it.dropna(subset=["Passenger Number"])), "2020-04-01", "2022-03-01")   # scheduled international flights suspended

# PRE-COVID BASELINE: April 2017 to March 2020 = three full years ending just before the lockdown, so every calendar month counts three times.
# The baseline for a month is the average of that calendar month over the three years ("seasonal baseline").
PRE = pd.date_range("2017-04-01", "2020-03-01", freq="MS")
seasonal = lambda Pm, k: Pm[k].loc[PRE].groupby(PRE.month).mean()            # 12 rows (month 1-12) x groups
BM = {k: seasonal(P, k) for k in P}
pre_share = P["pax"].loc[PRE].sum() / P["pax"].loc[PRE].sum().sum()
BASE = [g for g in P["pax"].columns if (P["pax"].loc[PRE, g] > 0).sum() == len(PRE) and pre_share[g] >= MIN_BASE_SHARE]   # full pre-COVID record
seas = lambda B, g: pd.Series(months.month, index=months).map(B[g])
RATIO = pd.DataFrame({g: P["pax"][g] / seas(BM["pax"], g) for g in BASE})          # passengers vs seasonal pre-COVID baseline
RATIO_RPK = pd.DataFrame({g: P["rpk"][g] / seas(BM["rpk"], g) for g in BASE})      # passenger-km vs seasonal pre-COVID baseline
ROLL = RATIO.rolling(3).mean()
MKB = mk_rpk.loc[PRE].groupby(PRE.month).mean()                                     # market passenger-km, seasonal baseline
FIRST = P["pax"].apply(lambda s: s[s > 0].index.min())
age_at = lambda g, T: (T.year - FIRST[g].year) * 12 + T.month - FIRST[g].month + 1

def status_of(g):   # recovery status uses passengers, as the project question asks about passenger numbers
    tr = P["pax"][g][P["pax"][g] > 0]
    if tr.empty: return "No traffic"
    operating = tr.index.max() == LAST
    if g in BASE:
        hit = ROLL.loc["2020-07-01":, g].dropna(); hit = hit[hit >= 1]
        if len(hit): return "Recovered and holding" if (operating and RATIO[g].iloc[-3:].mean() >= 1) else "Recovered, then fell back"
        return "Never recovered (still flying)" if operating else "Never recovered (stopped flying)"
    if tr.index.min() > PRE[-1]: return "New entrant (started after COVID)"
    return "No full pre-COVID record"
STATUS = {g: status_of(g) for g in P["pax"].columns}
print("groups with a full pre-COVID baseline:", BASE)
print(pd.Series(STATUS)[lambda s: s != "No traffic"].value_counts().to_string())


groups with a full pre-COVID baseline: ['Air India Group', 'Alliance Air', 'Go First', 'IndiGo', 'SpiceJet']
No full pre-COVID record             10
New entrant (started after COVID)     5
Recovered and holding                 2
Recovered, then fell back             2
Never recovered (still flying)        1


## 3. Seven metrics, scoring and countermeasures
Score = 50 + 25 x (value - median) / max(1.4826 x MAD, minimum spread), capped 0-100. Pillar = mean of its scores (missing = 50); index = mean of three pillars (missing = 50). `naive` reproduces the old method for comparison.

In [3]:
# THE 7 METRICS, measured "as of" a month T over the 12 months ending T (W) and the 12 months before (PW)
MET = {   # key: (label, unit, higher is better?, pillar)
    "dem":  ("Demand recovery",          "% of pre-COVID passenger-km",               True,  "Recovery"),
    "cap":  ("Capacity recovery",        "% of pre-COVID seat-km",                    True,  "Recovery"),
    "lf":   ("Load factor",              "% of seat-km filled",                       True,  "Operations"),
    "ppd":  ("Passengers per departure", "passengers",                                True,  "Operations"),
    "prod": ("Network productivity",     "thousand passenger-km per aircraft hour",   True,  "Operations"),
    "gro":  ("Traffic growth",           "log change in passenger-km vs previous 12 months (x100)", True, "Momentum"),
    "vol":  ("Volatility",               "std. dev. of monthly log year-on-year change (x100)", False, "Momentum"),
}
FORMULA = {
    "dem":  ("Sum of Passenger Kilometers over the last 12 months / average year of Apr 2017-Mar 2020 x 100", "Passenger Kilometers"),
    "cap":  ("Sum of Seat Kilometers over the last 12 months / average year of Apr 2017-Mar 2020 x 100", "Seat Kilometers"),
    "lf":   ("Sum of Passenger Kilometers / sum of Seat Kilometers x 100, last 12 months", "Passenger Kilometers, Seat Kilometers"),
    "ppd":  ("Sum of Passenger Number / sum of Aircraft Number (departures), last 12 months", "Passenger Number, Aircraft Number"),
    "prod": ("Sum of Passenger Kilometers (thousands) / sum of Aircraft Hours, last 12 months", "Passenger Kilometers, Aircraft Hours"),
    "gro":  ("ln(passenger-km, last 12 months / passenger-km, previous 12 months) x 100", "Passenger Kilometers"),
    "vol":  ("Std. dev. of ln(passenger-km this month / same month a year earlier) x 100; months whose year-earlier month had < 25,000 passengers are left out", "Passenger Kilometers, Passenger Number"),
}
WHY = {"dem": "Demand against the pre-COVID level, weighted by distance flown.", "cap": "Seats offered: how much the airline chose to supply.",
       "lf": "Share of seats filled.", "ppd": "Aircraft size times fullness: what each flight carries.", "prod": "How hard the aircraft work.",
       "gro": "Where traffic is heading.", "vol": "How erratic traffic is. Lower is better."}
PILLARS = ["Recovery", "Operations", "Momentum"]
MOM = [k for k in MET if MET[k][3] == "Momentum"]
FLOOR = {"dem": 10, "cap": 10, "lf": 2.5, "ppd": 15, "prod": 8, "gro": 5, "vol": 3}   # smallest spread treated as meaningful
lf_of = lambda rpk, ask: rpk / ask * 100

def metrics_asof(T, method="adjusted"):
    T = pd.Timestamp(T); i = months.get_loc(T)
    if i < 11: return pd.DataFrame(columns=list(MET))
    W = months[i - 11:i + 1]; PW = months[i - 23:i - 11] if i >= 23 else None
    rows = {}
    for g in P["pax"].columns:
        pw = P["pax"].loc[W, g]
        if not ((pw > 0).sum() == 12 and pw.mean() >= MIN_PAX): continue
        s = lambda k, ws=W: P[k].loc[ws, g].sum()
        r = dict(lf=lf_of(s("rpk"), s("ask")), ppd=s("pax") / s("dep"), prod=s("rpk") / s("hrs"))
        if PW is not None and (P["pax"].loc[PW, g] > 0).sum() == 12:
            ratio = s("rpk") / s("rpk", PW)
            r["gro"] = np.log(ratio) * 100 if method == "adjusted" else (ratio - 1) * 100
            now, before = P["rpk"].loc[W, g].values, P["rpk"].loc[PW, g].values
            ok = ((P["pax"].loc[PW, g].values >= MIN_PAX) if method == "adjusted" else np.ones(12, bool)) & (before > 0)
            if ok.sum() >= 6:
                ch = np.log(now[ok] / before[ok]) * 100 if method == "adjusted" else (now[ok] / before[ok] - 1) * 100
                r["vol"] = np.std(ch)
        if g in BASE:
            r["dem"] = s("rpk") / BM["rpk"][g].sum() * 100
            r["cap"] = s("ask") / BM["ask"][g].sum() * 100
        rows[g] = r
    return pd.DataFrame(rows).T.reindex(columns=list(MET)).astype(float)

def score(M, T, method="adjusted"):
    """adjusted: centre = median, spread = max(1.4826 x MAD, FLOOR); momentum pulled toward 50 for airlines younger than MATURE months;
    a missing pillar counts as 50. naive: mean and standard deviation, no damping, average of available pillars."""
    T = pd.Timestamp(T); S = pd.DataFrame(np.nan, index=M.index, columns=M.columns)
    for k in M.columns:
        x = M[k].dropna()
        if len(x) < 3: continue
        c, sc = (x.median(), max(1.4826 * (x - x.median()).abs().median(), FLOOR[k])) if method == "adjusted" else (x.mean(), x.std(ddof=0) or 1.0)
        z = ((x - c) / sc).clip(-2, 2)
        S.loc[x.index, k] = 50 + 25 * (z if MET[k][2] else -z)
    if method == "adjusted":
        for g in S.index:
            S.loc[g, MOM] = 50 + (S.loc[g, MOM] - 50) * min(1.0, age_at(g, T) / MATURE)
    Sx = S.fillna(50) if method == "adjusted" else S               # adjusted: a missing metric or pillar counts as 50 (no evidence = average)
    PIL = pd.DataFrame({p: Sx[[k for k in MET if MET[k][3] == p]].mean(axis=1) for p in PILLARS})
    measured = pd.DataFrame({p: S[[k for k in MET if MET[k][3] == p]].notna().any(axis=1) for p in PILLARS})
    if method == "adjusted": PIL[PILLARS] = PIL[PILLARS].where(measured)   # show a pillar with no data at all as n/a ...
    PIL["Index"] = PIL[PILLARS].fillna(50).mean(axis=1) if method == "adjusted" else PIL[PILLARS].mean(axis=1)   # ... but count it as 50
    PIL["Pillars measured"] = measured.sum(axis=1)
    return S, PIL

def reweigh(PIL, w):
    w = pd.Series(w); return (PIL[PILLARS].fillna(50) * w).sum(axis=1) / w.sum()

M0 = metrics_asof(LAST); S0, PIL0 = score(M0, LAST)
MN = metrics_asof(LAST, "naive"); SN, PILN = score(MN, LAST, "naive")
ORDER = PIL0.sort_values("Index", ascending=False).index.tolist()
HIST = {T: (lambda m: (m,) + score(m, T))(metrics_asof(T)) for T in months[months >= "2023-06-01"]}            # monthly, for the prediction tests
YEARS = [pd.Timestamp(f"{y}-12-01") for y in range(2018, LAST.year)] + [LAST]
YEARLY = {T: score(metrics_asof(T), T)[1] for T in YEARS}                                                      # the index at the end of every year
RKW = pd.DataFrame({n: reweigh(PIL0, w).rank(ascending=False) for n, w in {"equal": dict(Recovery=1, Operations=1, Momentum=1),
      "recovery x2": dict(Recovery=2, Operations=1, Momentum=1), "operations x2": dict(Recovery=1, Operations=2, Momentum=1),
      "momentum x2": dict(Recovery=1, Operations=1, Momentum=2)}.items()})
swing = RKW.max(axis=1) - RKW.min(axis=1)
print(f"airlines scored at {LAST:%b %Y}:", ORDER)
display(pd.concat([PIL0.loc[ORDER, PILLARS + ["Index"]].round(1), PILN.loc[ORDER, ["Index"]].round(1).rename(columns={"Index": "Index (old method)"})], axis=1))


airlines scored at Aug 2026: ['IndiGo', 'Akasa Air', 'Air India Group', 'Fly91', 'Star Air', 'SpiceJet', 'Alliance Air']


,Recovery,Operations,Momentum,Index,Index (old method)
IndiGo,70.4,57.2,56.5,61.4,69.4
Akasa Air,NaN,68.2,51.6,56.6,68.5
Air India Group,65.4,50.9,47.0,54.4,63.6
Fly91,NaN,13.2,70.8,44.7,42.5
Star Air,NaN,12.4,64.0,42.1,44.0
SpiceJet,31.7,57.2,28.6,39.2,44.5
Alliance Air,34.6,0.9,0.0,11.8,19.0


## 4. Prediction tests and the Go First backtest

In [4]:
# PREDICTION TESTS: each uses only information available at the time, then looks forward
spearman = lambda a, b: a.rank().corr(b.rank())
rows = []
for T, (Mt, St, Pt) in HIST.items():
    i = months.get_loc(T)
    if i + 6 >= len(months) or len(Mt) < 5: continue
    fut, prev = months[i + 1:i + 7], months[i - 11:i - 5]
    for g in Mt.index:   # outcome: passenger-km in the next 6 months vs the same 6 months a year earlier
        rows.append(dict(T=T, g=g, y=np.log(P["rpk"].loc[fut, g].sum() / P["rpk"].loc[prev, g].sum()) * 100, Index=Pt.loc[g, "Index"],
                         **{k: Mt.loc[g, k] for k in MET}, **{p: Pt.loc[g, p] for p in PILLARS}))
OBS = pd.DataFrame(rows)
PRED = pd.Series({c: spearman(OBS[[c, "y"]].dropna()[c], OBS[[c, "y"]].dropna()["y"]) for c in list(MET) + PILLARS + ["Index"]})
for k in MET:
    if not MET[k][2]: PRED[k] = -PRED[k]          # + means the metric's 'better' direction goes with faster growth
NOBS = OBS[list(MET) + PILLARS + ["Index"]].notna().sum()
print(f"{len(OBS)} airline-months. Rank correlation with the next 6 months of passenger-km growth:"); print(PRED.round(2).sort_values(ascending=False).to_string())

# Backtest: Go First stopped flying on 3 May 2023, so May 2023 is a near-empty month (48,296 passengers vs about 830,000).
# Scoring "as of May 2023" would let that partial month drive the result; the fair test is April 2023, its last full month.
BT_T = pd.Timestamp("2023-04-01")
BT_M = metrics_asof(BT_T); BT_S, BT_P = score(BT_M, BT_T)
BT_rank = BT_P["Index"].rank(ascending=False).astype(int)
BT_bad = score(metrics_asof("2023-05-01"), pd.Timestamp("2023-05-01"))[1]     # kept only to report the difference
print(f"\nBacktest as of {BT_T:%b %Y}: Go First ranks {BT_rank['Go First']} of {len(BT_P)} (index {BT_P.loc['Go First', 'Index']:.1f}); "
      f"using the partial May 2023 month it would rank {int(BT_bad['Index'].rank(ascending=False)['Go First'])}")

def forecast(s, T, h=6):    # same month last year x (last 12 months / the 12 months before)
    i = s.index.get_loc(T); g = s.iloc[i - 11:i + 1].sum() / s.iloc[i - 23:i - 11].sum()
    return pd.Series([s.iloc[i + k - 12] * g for k in range(1, h + 1)])
errs = {h: [] for h in range(1, 7)}
for T in months[(months >= "2024-08-01") & (months <= months[-7])]:
    f = forecast(mk_rpk, T)
    for h in range(1, 7): errs[h].append(abs(f.iloc[h - 1] / mk_rpk.iloc[months.get_loc(T) + h] - 1))
MAPE = pd.Series({h: np.mean(e) * 100 for h, e in errs.items()})
print("Projection error by months ahead (%):", MAPE.round(1).to_dict())


190 airline-months. Rank correlation with the next 6 months of passenger-km growth:
gro           0.75
Recovery      0.58
Momentum      0.51
cap           0.39
Index         0.39
dem           0.38
ppd           0.27
vol           0.12
prod          0.01
Operations    0.01
lf           -0.02

Backtest as of Apr 2023: Go First ranks 3 of 5 (index 39.6); using the partial May 2023 month it would rank 5
Projection error by months ahead (%): {1: 4.6, 2: 5.1, 3: 5.7, 4: 5.7, 5: 6.1, 6: 6.4}


## 5. City-pair data: load and clean

In [5]:
# CITY-PAIR DATA (second source): load, clean, log every step with before/after counts
gh_dir = RAW / "github"; gh_dir.mkdir(parents=True, exist_ok=True)
files = sorted(gh_dir.glob("vonter_domestic_city_*.csv"))
if files:
    CITY_FILE = files[-1]
else:   # download once, from the same GitHub commit the collection notebook recorded, and keep it as a raw file
    meta = json.load(open(latest(INTERIM, "collection_metadata_*.json"))); sha = meta.get("github_commit", "main")
    url = f"https://raw.githubusercontent.com/Vonter/india-aviation-traffic/{sha}/aggregated/domestic/city.csv"
    r = requests.get(url, headers={"User-Agent": "DA2402 data curation course project"}, timeout=60); r.raise_for_status()
    CITY_FILE = gh_dir / f"vonter_domestic_city_{TODAY}.csv"; CITY_FILE.write_bytes(r.content)
cp = pd.read_csv(CITY_FILE)
CLOG = [("raw file " + CITY_FILE.name, len(cp), len(cp), f"{cp.shape[1]} columns; Year, Month, City1, City2, PaxToCity2, PaxFromCity2, freight and mail")]
print("raw shape:", cp.shape); print(cp.dtypes.to_string())

CITY_MAP = {"ALLAHABAD": "PRAYAGRAJ", "ALLAHABAD (PRAYAGRAJ)": "PRAYAGRAJ", "BHATINDA": "BATHINDA", "COCHIN": "KOCHI", "CUDDAPAH": "KADAPA",
    "CUDDAPAH (KADAPA)": "KADAPA", "DEHRA DUN": "DEHRADUN", "DABOLIM": "GOA", "GOA (DABOLIM, SOUTH GOA)": "GOA", "GOA (MOPA, NORTH GOA)": "GOA",
    "MOPA, GOA": "GOA", "GULBARGA": "KALABURAGI", "HISSAR": "HISAR", "HOLLONGI, ITANAGAR": "ITANAGAR", "HIRASAR (RAJKOT)": "RAJKOT",
    "MANGALORE (MANGALURU)": "MANGALORE", "MUMBAI (MUMBAI)": "MUMBAI", "MUMBAI (NAVI MUMBAI)": "MUMBAI", "NASIK": "NASHIK",
    "PONDICHERRY": "PUDUCHERRY", "PONDICHERRY (PUDUCHERRY)": "PUDUCHERRY", "PURNEA": "PURNIA", "SIMLA": "SHIMLA", "SHOLAPUR": "SOLAPUR",
    "TIRUCHIRAPALLY": "TIRUCHIRAPPALLI", "TRIVANDRUM": "THIRUVANANTHAPURAM", "VIJAYWADA": "VIJAYAWADA",
    "VISHAKHAPATNAM (VISAKHAPATNAM)": "VISAKHAPATNAM", "ZERO": "ZIRO", "HINDON": "GHAZIABAD"}
import re
def canon(x):   # rule: trim spaces (incl. non-breaking), upper-case, drop 'AIRPORT' / 'INTERNATIONAL AIRPORT' / ', KARNATAKA', then map old and alternative names
    x = re.sub(r"\s+", " ", str(x).replace("\xa0", " ")).strip().upper()
    x = re.sub(r",\s*KARNATAKA$", "", x); x = re.sub(r"\s*(INTERNATIONAL )?AIRPORT$", "", x).strip()
    return CITY_MAP.get(x, x)
n_names = len(set(cp.City1) | set(cp.City2))
cp["A"], cp["B"] = cp.City1.map(canon), cp.City2.map(canon)
CLOG.append(("standardise city names (spaces, case, 'AIRPORT', renamed cities, two Goa airports, Navi Mumbai)", n_names, len(set(cp.A) | set(cp.B)), "unit = distinct city names"))
n = len(cp); cp = cp[cp.A != cp.B]; CLOG.append(("drop rows whose two ends became the same city", n, len(cp), "rows"))
cp["pax"] = cp.PaxToCity2 + cp.PaxFromCity2                               # both directions = passengers on the route
cp["Date"] = pd.to_datetime(dict(year=cp.Year, month=cp.Month, day=1))
cp["p1"], cp["p2"] = np.where(cp.A < cp.B, cp.A, cp.B), np.where(cp.A < cp.B, cp.B, cp.A)
dup = int(cp.duplicated(["Date", "City1", "City2"]).sum())
n_dir = len(set(zip(cp.City1, cp.City2))); n_pairs = len(set(zip(cp.p1, cp.p2)))
CR = cp.groupby(["Date", "p1", "p2"], as_index=False).pax.sum()             # one row per route per month
CLOG.append((f"merge A-to-B and B-to-A, renamed-city rows and {dup} split records (same route and month) into one route", len(cp), len(CR), f"rows; {n_dir:,} directional pairs -> {n_pairs:,} routes"))
LASTC = min(CR.Date.max(), LAST); CR = CR[CR.Date <= LASTC]
WC = pd.date_range(LASTC - pd.DateOffset(months=11), LASTC, freq="MS")
METRO = ["DELHI", "MUMBAI", "BENGALURU", "HYDERABAD", "CHENNAI", "KOLKATA"]
CR["kind"] = np.select([CR.p1.isin(METRO) & CR.p2.isin(METRO), CR.p1.isin(METRO) | CR.p2.isin(METRO)], ["Metro to metro", "Metro to other city"], "Other city to other city")
ctot = CR.groupby("Date").pax.sum()
XCHK = (ctot / mk.reindex(ctot.index)).dropna()
XCHK = XCHK[XCHK.index >= "2017-01-01"]
display(pd.DataFrame(CLOG, columns=["step", "before", "after", "note"]))
print(f"city-pair data ends {LASTC:%B %Y}; city-pair total / airline total: median {XCHK.median():.3f}, range {XCHK.min():.3f} to {XCHK.max():.3f} over {len(XCHK)} months")


raw shape: (66453, 10)
Year                  int64
Month                 int64
City1                   str
City2                   str
PaxToCity2          float64
PaxFromCity2        float64
FreightToCity2      float64
FreightFromCity2    float64
MailToCity2         float64
MailFromCity2       float64


,step,before,after,note
0,raw file vonter_domestic_city_2026-11-01.csv,66453,66453,"10 columns; Year, Month, City1, City2, PaxToCi..."
1,"standardise city names (spaces, case, 'AIRPORT...",194,144,unit = distinct city names
2,drop rows whose two ends became the same city,66453,66448,rows
3,"merge A-to-B and B-to-A, renamed-city rows and...",66448,65764,"rows; 2,183 directional pairs -> 1,470 routes"


city-pair data ends July 2026; city-pair total / airline total: median 1.000, range 0.994 to 1.046 over 112 months


## 6. Chart registry, text panels and worked example
`put()` registers a chart with its notes; `text_fig()` turns text into a Plotly figure.

In [6]:
# REGISTRY: every element of the page is a Plotly figure. Each chart has its own title and a notes panel
# (how it works / what it tells us / observations) that is also a Plotly figure.
FIGS = {}
SEC = {1: "The data: before and after cleaning", 2: "The seven metrics", 3: "Recovery against the pre-COVID level", 4: "Operations", 5: "Momentum and market share",
       6: "The route network (city-pair data)", 7: "International flights", 8: "Relationships and prediction", 9: "The scorecard", 10: "Conclusions", 11: "Other plots"}
COL = {"IndiGo": "#0072B2", "Air India Group": "#D55E00", "SpiceJet": "#E69F00", "Akasa Air": "#009E73", "Star Air": "#56B4E9",
       "Alliance Air": "#CC79A7", "Fly91": "#A0522D", "Go First": "#6A3D9A", "TruJet": "#8E44AD", "Jet Airways": "#333333", "Others": "#BDBDBD"}
SYM = {"IndiGo": "circle", "Air India Group": "square", "SpiceJet": "diamond", "Akasa Air": "triangle-up", "Star Air": "x",
       "Alliance Air": "star", "Fly91": "hexagon", "Go First": "cross", "TruJet": "pentagon", "Jet Airways": "triangle-down"}
gc = lambda g: COL.get(g, "#999999"); gs = lambda g: SYM.get(g, "circle")
def tint(h, f=0.45):
    r, g_, b_ = (int(h[i:i + 2], 16) for i in (1, 3, 5)); return "#%02x%02x%02x" % tuple(int(c + (255 - c) * f) for c in (r, g_, b_))
DIV = [[0, "#B35806"], [0.5, "#F2F2F2"], [1, "#2166AC"]]
bn = lambda v: v / 1e6
FONT = dict(family="Arial, sans-serif", size=13, color="#222")
WRAP = 165                                               # characters per line in text panels

def put(key, sec, title, fig, h=470, sub="", how="", tells="", obs=()):
    """Register a chart. title/sub go into the figure; how/tells/obs become the notes panel under it."""
    m = fig.layout.margin
    fig.update_layout(template="plotly_white", font=FONT, height=h,
                      title=dict(text=f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:#666'>{sub}</span>" if sub else ""), x=0.01, xanchor="left", xref="container", y=1, yref="container", yanchor="top", pad=dict(t=14), font=dict(size=17)),
                      margin=dict(l=60 if m.l is None else m.l, r=30 if m.r is None else m.r, t=max(m.t or 0, 95 if sub else 70), b=55 if m.b is None else m.b),
                      legend=dict(orientation="h", y=-0.2, x=0) if fig.layout.legend.y is None else fig.layout.legend)
    FIGS[key] = dict(sec=sec, title=title, fig=fig, how=how, tells=tells, obs=list(obs))

def wrap(t, w=WRAP): return "<br>".join(textwrap.wrap(t, w)) if t else ""
import re as _re
def _wrap_vis(t, w):
    """Wrap by visible characters, so HTML tags inside the text do not count towards the line length."""
    t = _re.sub("<[^>]+>", lambda m: m.group(0).replace(" ", "\x00"), t)      # keep each HTML tag in one piece
    out, cur, n = [], [], 0
    for word in t.split(" "):
        L = len(_re.sub("<[^>]+>", "", word))
        if cur and n + 1 + L > w: out.append(" ".join(cur)); cur, n = [], 0
        cur.append(word); n += (1 if n else 0) + L
    if cur: out.append(" ".join(cur))
    return [x.replace("\x00", " ") for x in out] or [""]
def text_fig(blocks, size=13, pad=14, bg="#f4f7fb", border="#2166AC", gap=False):
    """A Plotly figure that is only text: blocks = list of (heading, text or list of bullet strings)."""
    lines = []
    for bi, (head, body) in enumerate(blocks):
        if not body: continue
        if lines and gap: lines.append("")
        if head: lines.append(f"<b>{head}</b>")
        bul = isinstance(body, (list, tuple))
        for it_ in (body if bul else [body]):
            w_ = _wrap_vis(it_, WRAP - 4)
            lines += [("• " if bul else "") + w_[0]] + [("  " if bul else "") + x for x in w_[1:]]
    f = go.Figure()
    f.add_annotation(x=0, y=1, xref="paper", yref="paper", xanchor="left", yanchor="top", xshift=10, align="left", showarrow=False, captureevents=True, text="<br>".join(lines), font=dict(size=size, color="#222"))
    f.update_layout(meta="text", template="plotly_white", height=int(len(lines) * size * 1.42 + 2 * pad + 4), margin=dict(l=pad, r=pad, t=pad, b=pad), plot_bgcolor=bg, paper_bgcolor=bg,
                    dragmode=False, hovermode=False, xaxis=dict(visible=False, fixedrange=True), yaxis=dict(visible=False, fixedrange=True), shapes=[dict(type="line", xref="paper", yref="paper", x0=0, x1=0, y0=0, y1=1, line=dict(color=border, width=4))])
    return f
def notes_fig(v): return text_fig([("How it works", v["how"]), ("What it tells us", v["tells"]), ("Observations for the presentation", v["obs"])])

COMMON = ["lf", "ppd", "prod", "gro", "vol"]
top, bot = ORDER[0], ORDER[-1]
W12 = months[-12:]
SHORTU = {"dem": "% of pre-COVID", "cap": "% of pre-COVID", "lf": "% of seat-km", "ppd": "passengers", "prod": "000 pax-km per hour", "gro": "log points", "vol": "log points"}
def worked(g, T=LAST):
    i = months.get_loc(T); W, PW = months[i - 11:i + 1], months[i - 23:i - 11]
    s = lambda k, ws=W: P[k].loc[ws, g].sum(); v = M0.loc[g]
    return {"dem": f"{bn(s('rpk')):.1f} bn / {bn(BM['rpk'][g].sum()):.1f} bn = {v['dem']:.0f}%", "cap": f"{bn(s('ask')):.1f} bn / {bn(BM['ask'][g].sum()):.1f} bn = {v['cap']:.0f}%",
            "lf": f"{bn(s('rpk')):.1f} bn / {bn(s('ask')):.1f} bn = {v['lf']:.1f}%", "ppd": f"{s('pax'):,.0f} / {s('dep'):,.0f} = {v['ppd']:.0f}",
            "prod": f"{s('rpk'):,.0f} / {s('hrs'):,.0f} = {v['prod']:.0f}", "gro": f"ln({bn(s('rpk')):.1f} / {bn(s('rpk', PW)):.1f}) x 100 = {v['gro']:+.1f}",
            "vol": f"spread of 12 monthly changes = {v['vol']:.1f}"}
WORKED = worked("IndiGo")


## 7. Metrics and scorecard

In [7]:
BASE_IN = [g for g in ORDER if g in BASE]
# ---------------- SECTION 1: THE DATA, BEFORE AND AFTER CLEANING ----------------
clog = pd.read_csv(latest(CLEAN, "cleaning_log_*.csv"))
flow = [("Parsed from 142 Excel files", int(clog.rows_before.iloc[0]))] + [(r.step[0].upper() + r.step[1:], int(r.rows_after)) for r in clog.iloc[:3].itertuples()]
CORE = ["Aircraft Number", "Aircraft Hours", "Passenger Number", "Passenger Kilometers", "Seat Kilometers", "Passenger Load Factor"]
miss_b = interim[CORE].isna().mean() * 100; miss_a = clean[CORE].isna().mean() * 100
nb_, na_ = int(interim[CORE].isna().sum().sum()), int(clean[CORE].isna().sum().sum())
fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], horizontal_spacing=0.2,
                    subplot_titles=["Rows left after each cleaning step", "Share of cells missing, before vs after (%)"])
fig.add_trace(go.Bar(y=[f for f, _ in flow], x=[n for _, n in flow], orientation="h", marker=dict(color=["#B35806", "#9ecae1", "#6baed6", "#2166AC"]),
                     text=[f"{n:,}" for _, n in flow], textposition="outside", cliponaxis=False, showlegend=False, hovertemplate="%{y}: %{x:,} rows<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(y=CORE, x=miss_b.values, orientation="h", name=f"Before: parsed table ({len(interim):,} rows)", marker=dict(color="#B35806"),
                     text=[f"{v:.0f}%" for v in miss_b], textposition="outside", cliponaxis=False, hovertemplate="%{y}: %{x:.1f}% missing before<extra></extra>"), row=1, col=2)
fig.add_trace(go.Bar(y=CORE, x=miss_a.values, orientation="h", name=f"After: clean table ({len(clean):,} rows)", marker=dict(color="#2166AC"),
                     text=[f"{v:.1f}%" for v in miss_a], textposition="outside", cliponaxis=False, hovertemplate="%{y}: %{x:.2f}% missing after<extra></extra>"), row=1, col=2)
fig.update_yaxes(autorange="reversed", row=1, col=1); fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_xaxes(range=[0, len(interim) * 1.18], row=1, col=1); fig.update_xaxes(range=[0, 65], row=1, col=2)
fig.update_layout(barmode="group", bargap=0.25, margin=dict(l=40, r=30), legend=dict(orientation="h", y=-0.15, x=0.45)); fig.update_annotations(font_size=13)
lf0 = int(clean["Passenger Load Factor"].isna().sum()); lfz = int((clean["Passenger Load Factor"].isna() & (clean["Seat Kilometers"] == 0)).sum())
put("cleaning", 1, "What cleaning did to the data", fig, 440, "Left: rows through the cleaning steps. Right: missing cells in the six core columns",
    how="Left: how many rows remain after each logged cleaning step, starting from everything parsed out of DGCA's Excel files. Right: for each core column, the share of empty cells in the parsed table (orange) and in the clean table (blue).",
    tells="That most of the 'missing data' was never data at all: DGCA's sheets list every airline in every section, and rows for airlines that did not fly are left blank.",
    obs=[f"Removing {int(clog.rows_affected.iloc[0]):,} completely empty template rows takes the core columns from about {miss_b.drop('Passenger Load Factor').mean():.0f}% missing to under 1%: {nb_:,} empty cells before, {na_:,} after.",
         f"Load factor is the only column with a visible gap ({lf0} rows), and {lfz} of those are months with zero passengers and zero seats, where load factor is 0/0; the other {lf0 - lfz} are rows where the kilometres themselves are missing. We kept the zeros, left those blank and imputed nothing; the analysis recomputes load factor from 12-month totals anyway."])

# ---------------- SECTION 2: THE SEVEN METRICS ----------------
# Plotly tables do not wrap a cell that already contains a line break, so every cell is wrapped here to fit its column.
TW = [70, 190, 300, 165, 55, 205]; TCH = [11, 30, 48, 26, 8, 32]          # column widths and characters per line
tcols = [[MET[k][3] for k in MET], [[f"<b>{MET[k][0]}</b>", WHY[k]] for k in MET], [FORMULA[k][0] for k in MET], [FORMULA[k][1] for k in MET],
         ["higher" if MET[k][2] else "lower" for k in MET], [WORKED[k] for k in MET]]
def tcell(v, w):
    parts = v if isinstance(v, list) else [v]
    return [ln for part in parts for ln in _wrap_vis(part, w)]
tlines = [[tcell(v, TCH[j]) for v in col] for j, col in enumerate(tcols)]
row_h = [max(len(tlines[j][i]) for j in range(len(tcols))) * 17 + 14 for i in range(len(MET))]
MET_TABLE = go.Figure(go.Table(columnwidth=TW,
    header=dict(values=["<b>Pillar</b>", "<b>Metric</b>", "<b>Formula (last 12 months)</b>", "<b>Columns used</b>", "<b>Better</b>", "<b>IndiGo, worked</b>"], fill_color="#222", font=dict(color="white", size=12), align="left", height=30),
    cells=dict(values=[["<br>".join(c) for c in col] for col in tlines], align="left", font=dict(size=11.5), height=max(row_h), fill_color=[["#ffffff", "#f6f8fb"] * 4])))
MET_TABLE.update_layout(template="plotly_white", font=FONT, height=30 + sum(max(h, max(row_h)) for h in row_h) + 100, margin=dict(l=10, r=10, t=60, b=10),
                        title=dict(text="<b>How each metric is calculated</b><br><span style='font-size:12px;color:#666'>Pre-COVID level = April 2017 to March 2020, each month compared with the same calendar month averaged over those three years. Passenger-km and seat-km are in thousands.</span>", x=0.01, font=dict(size=17)))
METHOD_TEXT = text_fig([("From value to score to index", ["Score = 50 + 25 x (value - median of the scored airlines) / spread, capped to 0-100; volatility is flipped because lower is better.",
                                                           "Pillar = average of its metric scores; index = average of the three pillars (Recovery, Operations, Momentum).",
                                                           f"An airline is scored if it flew all of the last 12 months with at least {MIN_PAX:,} passengers a month."]),
                        ("Countermeasures for new airlines", ["Momentum needs two full years of history.", f"Growth is a log change, so a ramp-up from a tiny base does not explode (Fly91: {MN.loc['Fly91', 'gro']:+.0f}% becomes {M0.loc['Fly91', 'gro']:+.0f} log points).",
                                                              "Launch months are left out of volatility.", f"Momentum scores are pulled toward 50 until an airline is {MATURE} months old.",
                                                              "Scores use the median and a minimum meaningful spread, so one extreme airline cannot stretch the scale.", "Missing evidence (no pre-COVID record, or a metric that cannot be measured) counts as 50."])])

seen = set()
fig = make_subplots(rows=2, cols=4, subplot_titles=[f"{MET[k][0]}<br><sup>{SHORTU[k]}</sup>" for k in MET], shared_yaxes=True, horizontal_spacing=0.035, vertical_spacing=0.2)
for j, k in enumerate(MET):
    r, c = j // 4 + 1, j % 4 + 1; x = M0[k].dropna()
    for g in ORDER:
        if g in x.index:
            fig.add_trace(go.Scatter(x=[x[g]], y=[g], mode="markers", marker=dict(color=gc(g), symbol=gs(g), size=11, line=dict(color="white", width=1)),
                                     name=g, legendgroup=g, showlegend=g not in seen, hovertemplate=f"{g}<br>{MET[k][0]}: %{{x:.1f}}<extra></extra>"), row=r, col=c); seen.add(g)
    if len(x): fig.add_vline(x=float(x.median()), line=dict(dash="dot", color="#888", width=1), row=r, col=c)
fig.update_yaxes(categoryorder="array", categoryarray=ORDER[::-1], tickfont=dict(size=11)); fig.update_xaxes(tickfont=dict(size=10)); fig.update_annotations(font_size=11)
fig.update_layout(margin=dict(l=120, r=20, b=40), legend=dict(orientation="h", y=-0.12))
turbo = [g for g in ORDER if M0.loc[g, "ppd"] < 100]
put("m_dots", 2, "Each metric, airline by airline", fig, 640, "Raw values before scoring; dotted line = median of the scored airlines",
    how="One small panel per metric. Each symbol is an airline at its raw value over the last 12 months; the dotted line is the median of the seven scored airlines. Recovery panels only show airlines with a full pre-COVID record.",
    tells="Where every airline stands on each metric before anything is converted to a score, so the index can be traced back to real numbers.",
    obs=[f"Only IndiGo ({M0.loc['IndiGo', 'dem']:.0f}%) and Air India Group ({M0.loc['Air India Group', 'dem']:.0f}%) fly far more passenger-km than before COVID; SpiceJet flies {M0.loc['SpiceJet', 'dem']:.0f}% of its pre-COVID level.",
         f"The turboprop airlines ({', '.join(turbo)}) sit together at the bottom of all three operations panels: {M0.loc[turbo, 'ppd'].min():.0f} to {M0.loc[turbo, 'ppd'].max():.0f} passengers per flight against {M0.loc[[g for g in ORDER if g not in turbo], 'ppd'].min():.0f} or more for the jets. That is aircraft size, not management."])

cmp = pd.DataFrame({"old": PILN["Index"], "new": PIL0["Index"]}).loc[ORDER]; dch = (cmp.new - cmp.old)
fig = go.Figure()
for g in ORDER:
    fig.add_trace(go.Scatter(x=[cmp.loc[g, "old"], cmp.loc[g, "new"]], y=[g, g], mode="lines", line=dict(color="#c8c8c8", width=5), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=cmp.old, y=ORDER, mode="markers", name="Old method", marker=dict(symbol="circle-open", size=13, color="#555", line=dict(width=2.5))))
fig.add_trace(go.Scatter(x=cmp.new, y=ORDER, mode="markers", name="With countermeasures", marker=dict(symbol="diamond", size=13, color="#0072B2")))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title="Performance index (0 to 100)", range=[0, 90]); fig.update_layout(margin=dict(l=130))
put("m_counter", 2, "What the countermeasures for new airlines change", fig, 420,
    how="Each line joins an airline's index under the old method (circle: mean and standard deviation, % growth, no age damping, new airlines averaged over the pillars they have) and the new method (diamond).",
    tells="Whether the ranking depends on how new airlines are treated.",
    obs=[f"Akasa Air falls the most ({dch['Akasa Air']:+.0f} points, from {cmp.loc['Akasa Air', 'old']:.0f} to {cmp.loc['Akasa Air', 'new']:.0f}): the old method scored it only on its two strongest pillars and put it first.",
         f"With the countermeasures {top} leads ({PIL0.loc[top, 'Index']:.0f}); every airline with a pre-COVID record loses {(PILN.loc[BASE_IN, 'Index'] - PIL0.loc[BASE_IN, 'Index']).min():.0f} to {(PILN.loc[BASE_IN, 'Index'] - PIL0.loc[BASE_IN, 'Index']).max():.0f} points, because scores are now centred on the median and capped."])

# ---------------- SECTION 2: THE SCORECARD ----------------
cols_ = ["Index", "Recovery", "Operations", "Momentum"]; Z = PIL0.loc[ORDER, cols_]
fig = go.Figure(go.Heatmap(z=Z.values, x=cols_, y=ORDER, colorscale="Viridis", zmin=0, zmax=100, xgap=3, ygap=3,
                           text=[[("n/a" if pd.isna(v) else f"{v:.0f}") for v in row] for row in Z.values], texttemplate="%{text}", textfont=dict(size=14),
                           colorbar=dict(title=dict(text="Score"), tickvals=[0, 25, 50, 75, 100], ticktext=["0 weak", "25", "50 typical", "75", "100 strong"]), hovertemplate="%{y}<br>%{x}: %{text}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(side="top"); fig.update_layout(margin=dict(l=130))
put("scorecard", 9, "The scorecard", fig, 420,
    how="Rows are airlines ordered by index; columns are the index and its three pillars. Colour and number are the same score (0 weak, 50 typical, 100 strong). n/a = no pre-COVID record; it counts as 50 in the index.",
    tells="The overall ranking and which pillar drives each airline's position.",
    obs=[f"{top} is the only airline above 55 on all three pillars; {bot} is last ({PIL0.loc[bot, 'Index']:.0f}) and weak on every pillar.",
         f"Fly91 has the strongest momentum ({PIL0.loc['Fly91', 'Momentum']:.0f}) but very weak operations ({PIL0.loc['Fly91', 'Operations']:.0f}): a small, fast-growing regional airline. SpiceJet is the opposite: decent operations ({PIL0.loc['SpiceJet', 'Operations']:.0f}) but the weakest recovery among the big airlines ({PIL0.loc['SpiceJet', 'Recovery']:.0f})."])

th = [MET[k][0] for k in MET]
def radar_vals(g):
    r_ = [float(S0.loc[g, k]) if pd.notna(S0.loc[g, k]) else 50.0 for k in MET]; return r_ + r_[:1]
hov = lambda g: [f"{g}<br>{MET[k][0]}: " + (f"{S0.loc[g, k]:.0f}" if pd.notna(S0.loc[g, k]) else "no data (shown as 50)") for k in MET] + [""]
A0, B0 = "IndiGo", "SpiceJet"
fig = go.Figure([go.Scatterpolar(r=radar_vals(g), theta=th + th[:1], name=g, fill="toself", opacity=.35, line=dict(color=gc(g), width=3), fillcolor=gc(g),
                                 mode="lines+markers", marker=dict(symbol=gs(g), size=8), text=hov(g), hoverinfo="text") for g in (A0, B0)])
def menu(i, x, g0):
    return dict(type="dropdown", x=x, y=1.12, xanchor="left", showactive=True, active=ORDER.index(g0),
                buttons=[dict(label=g, method="restyle", args=[{"r": [radar_vals(g)], "name": [g], "text": [hov(g)], "line.color": [gc(g)], "fillcolor": [gc(g)], "marker.symbol": [gs(g)]}, [i]]) for g in ORDER])
fig.update_layout(updatemenus=[menu(0, 0.1, A0), menu(1, 0.6, B0)], polar=dict(radialaxis=dict(range=[0, 100], tickvals=[25, 50, 75])), margin=dict(t=110),
                  annotations=[dict(text="Airline 1", x=0, y=1.1, xref="paper", yref="paper", showarrow=False, xanchor="left"), dict(text="Airline 2", x=0.5, y=1.1, xref="paper", yref="paper", showarrow=False, xanchor="left")])
put("radar", 9, "Compare two airlines", fig, 560, "Pick any two airlines from the menus",
    how="Each of the seven spokes is a metric score (centre 0, outer edge 100). The two menus choose which airlines are drawn; a metric an airline does not have is drawn at 50.",
    tells="The shape of each airline's strengths, so two airlines can be compared metric by metric.",
    obs=[f"IndiGo vs SpiceJet: their operations scores are almost the same (load factor {S0.loc['IndiGo', 'lf']:.0f} vs {S0.loc['SpiceJet', 'lf']:.0f}, productivity {S0.loc['IndiGo', 'prod']:.0f} vs {S0.loc['SpiceJet', 'prod']:.0f}); the difference is all in recovery and stability. SpiceJet's problem is how much it flies, not how well it flies.",
         f"Akasa Air vs IndiGo: Akasa beats IndiGo on load factor and passengers per flight but has no pre-COVID record, so its Recovery spokes sit at 50."])

YL = [str(T.year) if T.month == 12 else f"{T:%b %Y}" for T in YEARS]
RKY = pd.DataFrame({lbl: YEARLY[T]["Index"] for lbl, T in zip(YL, YEARS)}).T.rank(axis=1, ascending=False).dropna(how="all")
XL = list(RKY.index); xpos = {l: i for i, l in enumerate(XL)}          # positions 0..n-1; labels set as tick text (numeric-looking labels confused the axis)
fig = go.Figure()
for g in RKY.columns:
    s = RKY[g]
    fig.add_trace(go.Scatter(x=[xpos[l] for l in s.index], y=s.tolist(), mode="lines+markers", name=g, line=dict(color=gc(g), width=2.5), marker=dict(size=10, symbol=gs(g)), connectgaps=False,
                             customdata=list(s.index), hovertemplate=g + ": rank %{y:.0f} in %{customdata}<extra></extra>"))
    lv = s.dropna()
    if len(lv): fig.add_annotation(x=xpos[lv.index[-1]], y=float(lv.iloc[-1]), text=g, xanchor="left", xshift=10, showarrow=False, font=dict(size=11, color=gc(g)))
fig.update_xaxes(tickvals=list(range(len(XL))), ticktext=XL, range=[-0.4, len(XL) - 0.2], title="Index for the 12 months to December of each year (last point: to " + f"{LAST:%B %Y})")
fig.update_yaxes(autorange="reversed", title="Rank (1 = best)", dtick=1); fig.update_layout(showlegend=False, margin=dict(r=40))
r19 = RKY.loc["2019"].dropna().sort_values()
missing_years = [str(T.year) for T in YEARS if len(YEARLY[T]) == 0]
put("bump", 9, "Rank year by year since 2018", fig, 480, (f"{', '.join(missing_years)} is missing: no airline flew all 12 months that year" if missing_years else ""),
    how="Each point is an airline's rank on the index for the 12 months ending in December (last point: the latest 12 months). An airline appears only in years it flew all 12 months.",
    tells="How the pecking order changed from before COVID to now.",
    obs=[f"Before COVID the order was different: in 2019 {r19.index[0]} ranked first and {r19.index[1]} second. {r19.index[0]} stopped flying in 2023; SpiceJet fell to rank {int(RKY.iloc[-1]['SpiceJet'])}.",
         f"IndiGo ranks first in {int((RKY['IndiGo'] == 1).sum())} of {RKY['IndiGo'].notna().sum()} years; Air India Group has held a top-three place every year since 2021."])


## 8. Recovery

In [8]:
# ---------------- SECTION 3: RECOVERY AGAINST THE PRE-COVID LEVEL ----------------
POST = months >= "2020-04-01"
RATIO_ASK = pd.DataFrame({g: P["ask"][g] / seas(BM["ask"], g) for g in BASE})
rb = [g for g in ORDER if g in BASE] + [g for g in BASE if g not in ORDER]
nc = 3; nr = int(np.ceil(len(rb) / nc))
fig = make_subplots(rows=nr, cols=nc, subplot_titles=[f"<b>{g}</b><br><sup>{STATUS[g]}</sup>" for g in rb], shared_yaxes=True, shared_xaxes=True, vertical_spacing=0.2, horizontal_spacing=0.04)
for j, g in enumerate(rb):
    r, c = j // nc + 1, j % nc + 1
    for ser, nm, dash, w in [(RATIO_RPK[g], "Passenger-km (demand)", "solid", 2.6), (RATIO_ASK[g], "Seat-km (capacity offered)", "dot", 2)]:
        s = (ser.rolling(3).mean() * 100).loc["2017-06-01":]
        fig.add_trace(go.Scatter(x=s.index, y=s, mode="lines", line=dict(color=gc(g) if dash == "solid" else "#555", width=w, dash=dash), name=nm, legendgroup=nm, showlegend=(j == 0),
                                 hovertemplate=f"{g} {nm.split(' (')[0].lower()} %{{x|%b %Y}}: %{{y:.0f}}%<extra></extra>"), row=r, col=c)
    fig.add_hline(y=100, line=dict(color="#222", width=1, dash="dash"), row=r, col=c)
    fig.add_vrect(x0="2020-04-01", x1="2020-06-01", fillcolor="#e6e6e6", line_width=0, layer="below", row=r, col=c)
fig.update_yaxes(range=[0, 200], ticksuffix="%"); fig.update_xaxes(dtick="M24", tickformat="%Y"); fig.update_annotations(font_size=12)
fig.update_layout(legend=dict(orientation="h", y=-0.12, x=0))
pre_g = [g for g in P["pax"].columns if (P["pax"].loc[PRE, g] > 0).any()]
why_out = {}
for g in pre_g:
    if g in BASE: continue
    s = P["pax"].loc[PRE, g]; act = s[s > 0]
    why_out[g] = ("stopped flying in " + f"{act.index.max():%b %Y}") if act.index.max() < PRE[-1] else (("started in " + f"{act.index.min():%b %Y}") if act.index.min() > PRE[0] else f"too small ({pre_share[g] * 100:.2f}% of passengers)")
capdem = {g: (M0.loc[g, "dem"], M0.loc[g, "cap"]) for g in rb if g in M0.index}
put("rec_each", 3, "Each airline against its own pre-COVID level", fig, 260 * nr + 120, "3-month averages, as % of the same calendar month averaged over April 2017 to March 2020",
    how="One panel per airline. The coloured line is passenger-km (demand), the dotted line seat-km (capacity the airline chose to offer), both against the airline's own pre-COVID level (dashed line = 100%). The grey band is the April-May 2020 lockdown.",
    tells="Whether each airline got back to its pre-COVID size, and whether a gain or drop came from demand or from the airline adding or cutting capacity: if both lines move together, the airline changed its size; if the coloured line falls below the dotted one, planes are emptier.",
    obs=[f"Why only {len(BASE)} airlines: the comparison needs every month of April 2017 to March 2020 and at least 0.5% of passengers. Left out: " + "; ".join(f"{g} ({w})" for g, w in why_out.items()) + ". Akasa Air, Fly91 and others started after COVID.",
         f"Gains: IndiGo (demand {capdem['IndiGo'][0]:.0f}%, capacity {capdem['IndiGo'][1]:.0f}%) and Air India Group ({capdem['Air India Group'][0]:.0f}%, {capdem['Air India Group'][1]:.0f}%) grew by adding aircraft and absorbed the passengers of Jet Airways (gone 2019), Go First (gone 2023) and Vistara and AirAsia India (merged into the group).",
         f"Drops: SpiceJet's capacity fell to {capdem['SpiceJet'][1]:.0f}% and its demand to {capdem['SpiceJet'][0]:.0f}%, the two lines move together, so it shrank its own operation rather than losing passengers on the flights it kept (load factor still {M0.loc['SpiceJet', 'lf']:.0f}%). Alliance Air (capacity {capdem['Alliance Air'][1]:.0f}%) shows the same pattern. Go First recovered by May 2022 and then stopped flying in May 2023."])

show = ["IndiGo", "Air India Group", "SpiceJet", "Akasa Air", "Go First"]
ra = P["rpk"].loc[POST].fillna(0) / 1e6; other = ra.drop(columns=show).sum(axis=1)
fig = go.Figure()
for g in show[::-1] + ["Others"]:
    fig.add_trace(go.Scatter(x=ra.index, y=other if g == "Others" else ra[g], name=g, stackgroup="one", mode="lines", line=dict(width=0.5, color="white"), fillcolor=gc(g), hovertemplate=g + ": %{y:.2f} bn<extra></extra>"))
fig.add_trace(go.Scatter(x=ra.index, y=pd.Series(ra.index.month, index=ra.index).map(MKB) / 1e6, name="Pre-COVID level (same month)", mode="lines", line=dict(color="#111", dash="dash", width=1.5)))
fig.update_yaxes(title="Passenger-km per month (billions)"); fig.update_xaxes(title="Month")
mratio = (mk_rpk / pd.Series(months.month, index=months).map(MKB)).rolling(3).mean()
first_mkt = mratio.loc["2020-07-01":][lambda s: s >= 1].index.min()
put("rpk_post", 3, "The market since April 2020", fig, 480,
    how="Bands stack each airline's monthly passenger-km, so the top edge is the whole market. The dashed line is the market's pre-COVID level for the same calendar month.",
    tells="When the market as a whole recovered and who carried that recovery.",
    obs=[f"The market first held its pre-COVID level in {first_mkt:%B %Y} and is now {(mk_rpk.loc[W12].sum() / MKB.sum() - 1) * 100:+.0f}% above it.",
         "The two dips after the lockdown are the second wave (April-June 2021) and Omicron (January 2022). Go First's purple band ends in May 2023 and IndiGo's blue band fills most of the gap."])

act_base = [g for g in ORDER if g in BASE]
fig = go.Figure([go.Box(y=(RATIO_RPK.loc[POST, g] * 100).dropna(), name=g, marker=dict(color=gc(g), symbol=gs(g)), boxpoints="all", jitter=.4, pointpos=-1.7, boxmean=True) for g in act_base])
fig.add_hline(y=100, line=dict(dash="dash", color="#222"), annotation_text="pre-COVID level", annotation_position="top right")
fig.update_yaxes(title="Passenger-km, % of pre-COVID level"); fig.update_layout(showlegend=False)
med = {g: (RATIO_RPK.loc[POST, g] * 100).median() for g in act_base}; best_sj = (RATIO_RPK.loc[POST, "SpiceJet"] * 100).max()
put("box", 3, "Every month since April 2020", fig, 470,
    how="Each dot is one month since April 2020, compared with the same calendar month before COVID. The box covers the middle half of the months; the solid line is the median, the dashed line the mean.",
    tells="How typical and how variable each airline's months have been, not just its latest value.",
    obs=[f"IndiGo's typical month is {med['IndiGo']:.0f}% of its pre-COVID level; SpiceJet's is {med['SpiceJet']:.0f}%, and its best month since April 2020 reached only {best_sj:.0f}%.",
         "The long lower whiskers of every airline are the lockdown and second-wave months; the boxes show where they settled after that."])

# TWO TOWERS AND A PLANE: smooth animation (only the plane and the change panel move), with play / pause / restart
tw = ["IndiGo", "Air India Group", "Akasa Air", "SpiceJet", "Go First", "Jet Airways"]
L_ = P["rpk"].loc[PRE].fillna(0).mean() / 1e6; R_ = P["rpk"].loc[W12].fillna(0).mean() / 1e6
Lv = {g: L_[g] for g in tw}; Rv = {g: R_[g] for g in tw}; Lv["Others"] = L_.drop(tw).sum(); Rv["Others"] = R_.drop(tw).sum()
TL, TR = sum(Lv.values()), sum(Rv.values()); chg = {g: Rv[g] - Lv[g] for g in tw + ["Others"]}
fig = go.Figure()      # towers are drawn as layout shapes: bar traces vanish in Plotly when other traces animate without a full redraw
for x0, V in ((0, Lv), (2, Rv)):
    y0 = 0
    for g in tw + ["Others"]:
        v = V[g]
        if v <= 0: continue
        fig.add_shape(type="rect", x0=x0 - 0.3, x1=x0 + 0.3, y0=y0, y1=y0 + v, fillcolor=gc(g), line=dict(color="white", width=1.5), layer="above")
        if v > 0.6: fig.add_annotation(x=x0, y=y0 + v / 2, text=g, showarrow=False, font=dict(color="white", size=12))
        fig.add_trace(go.Scatter(x=[x0], y=[y0 + v / 2], mode="markers", marker=dict(size=40, opacity=0, symbol="square"), showlegend=False,
                                 hovertemplate=f"{g}: {v:.2f} bn passenger-km a month<extra></extra>"))
        y0 += v
fig.add_trace(go.Scatter(x=[0, 2], y=[TL, TR], mode="text", text=[f"<b>{TL:.1f} bn</b>", f"<b>{TR:.1f} bn</b>"], textposition="top center", showlegend=False, hoverinfo="skip"))
# The plane takes 7 steps, one per airline: each step moves it up or down by exactly that airline's change in passenger-km,
# so it starts on top of the old tower (old market) and lands on top of the new one (new market).
STEPS = sorted(chg.items(), key=lambda t: -t[1])                       # gains first, then losses
ALT = np.r_[TL, TL + np.cumsum([v for _, v in STEPS])]                 # altitude after each step = market size so far
XS_ = np.linspace(0.36, 1.45, len(STEPS) + 1)
def plane(i):
    return go.Scatter(x=[XS_[i]], y=[ALT[i]], mode="markers+text", marker=dict(size=9, color="#111"), text=[f"\u2708 {ALT[i]:.1f} bn"],
                      textposition="top center" if i == 0 else ("middle right" if STEPS[i - 1][1] >= 0 else "top right"), textfont=dict(size=19, color="#111"), showlegend=False, hoverinfo="skip")
def trail(i):    # always 8 points (future ones parked on the plane) so the transition is smooth
    k = [min(j, i) for j in range(len(ALT))]
    return go.Scatter(x=XS_[k], y=ALT[k], mode="lines", line=dict(color="#888", width=2, dash="dot", shape="hv"), showlegend=False, hoverinfo="skip")
def labels(i):
    k = list(range(1, len(ALT)))
    return go.Scatter(x=[XS_[j] for j in k], y=[ALT[j] for j in k], mode="text", textposition=["top center" if abs(STEPS[j - 1][1]) < 0.3 else ("top left" if STEPS[j - 1][1] > 0 else "bottom right") for j in k],
                      text=[(f"<b>{STEPS[j - 1][0]}</b><br>{STEPS[j - 1][1]:+.1f} bn" if j <= i else "") for j in k],
                      textfont=dict(size=12, color=[gc(STEPS[j - 1][0]) if STEPS[j - 1][0] != "Others" else "#666" for j in k]), showlegend=False, hoverinfo="skip")
fig.add_trace(trail(0)); fig.add_trace(labels(0)); fig.add_trace(plane(0)); T0 = len(fig.data) - 3
fname = ["Start"] + [g for g, _ in STEPS]
fig.frames = [go.Frame(name=fname[i], data=[trail(i), labels(i), plane(i)], traces=[T0, T0 + 1, T0 + 2]) for i in range(len(ALT))]
anim = lambda cur: dict(frame=dict(duration=1300, redraw=False), transition=dict(duration=650, easing="cubic-in-out"), fromcurrent=cur, mode="immediate")
fig.update_layout(showlegend=False, margin=dict(r=40, t=110, b=120),
                  xaxis=dict(range=[-0.45, 2.45], showgrid=False, zeroline=False, tickvals=[0, 2], ticktext=["Average month before COVID<br>(Apr 2017 to Mar 2020)", f"Average month<br>({W12[0]:%b %Y} to {W12[-1]:%b %Y})"]),
                  yaxis=dict(title="Passenger-km per month (billions)", range=[0, ALT.max() + 2.5]),
                  updatemenus=[dict(type="buttons", direction="right", x=0, y=1.13, xanchor="left", showactive=False,
                                    buttons=[dict(label="\u25B6 Play", method="animate", args=[None, anim(True)]),
                                             dict(label="\u23F8 Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), transition=dict(duration=0), mode="immediate")]),
                                             dict(label="\u21BA Restart", method="animate", args=[None, anim(False)])])],
                  sliders=[dict(x=0.05, y=-0.12, len=0.9, currentvalue=dict(prefix="Step: ", font=dict(size=12)), pad=dict(t=10),
                                steps=[dict(label=n_, method="animate", args=[[n_], dict(mode="immediate", frame=dict(duration=0, redraw=False), transition=dict(duration=500, easing="cubic-in-out"))]) for n_ in fname])])
put("towers", 3, "Who gained and who lost passenger-km", fig, 620, "Press Play (or drag the slider): the plane climbs or drops by each airline's change, one airline per step",
    how="Each tower stacks the airlines' passenger-km in an average month: before COVID on the left, the last 12 months on the right. The plane starts at the old market size and takes seven steps, one per airline: each step moves it up or down by exactly that airline's change in passenger-km a month, so it lands on the new market size. The dotted staircase keeps the path and each step is labelled.",
    tells="How the market grew and which airlines took the growth.",
    obs=[f"The average month grew from {TL:.1f} to {TR:.1f} billion passenger-km ({(TR / TL - 1) * 100:+.0f}%). IndiGo alone added {chg['IndiGo']:+.1f} billion, more than the whole market's net growth of {TR - TL:+.1f} billion.",
         f"Jet Airways and Go First took {chg['Jet Airways'] + chg['Go First']:+.1f} billion a month out of the market, and SpiceJet another {chg['SpiceJet']:+.1f}; IndiGo, Air India Group and Akasa Air filled that space."])

b = sd[sd.Date.isin(PRE)].groupby(["Airline", "Group"])["Passenger Number"].sum().div(len(PRE) / 12).reset_index(); b = b[b["Passenger Number"] > 0]
b["src"] = np.where(b["Passenger Number"] < 0.5e6, "Other small brands", b.Airline)
L = b.groupby(["src", "Group"])["Passenger Number"].sum().reset_index(); gout = L.groupby("Group")["Passenger Number"].sum()
OUTC = {"Recovered and holding": "#2166AC", "Recovered, then fell back": "#E69F00", "Never recovered (still flying)": "#B35806", "Never recovered (stopped flying)": "#6A3D9A",
        "New entrant (started after COVID)": "#7F7F7F", "No full pre-COVID record": "#BBBBBB"}
rgba = lambda h, a: f"rgba({int(h[1:3], 16)},{int(h[3:5], 16)},{int(h[5:7], 16)},{a})"
nodes = [("b", s) for s in L.src.unique()] + [("g", g) for g in gout.index] + [("o", o) for o in sorted({STATUS[g] for g in gout.index})]
nid = {n_: i for i, n_ in enumerate(nodes)}
fig = go.Figure(go.Sankey(arrangement="snap", node=dict(label=[n_ for _, n_ in nodes], color=["#bbbbbb" if t == "b" else (gc(n_) if t == "g" else OUTC[n_]) for t, n_ in nodes], pad=16, thickness=16),
        link=dict(source=[nid[("b", r.src)] for r in L.itertuples()] + [nid[("g", g)] for g in gout.index], target=[nid[("g", r.Group)] for r in L.itertuples()] + [nid[("o", STATUS[g])] for g in gout.index],
                  value=[v / 1e6 for v in L["Passenger Number"]] + [v / 1e6 for v in gout.values], color=["rgba(150,150,150,.35)"] * len(L) + [rgba(OUTC[STATUS[g]], .45) for g in gout.index],
                  hovertemplate="%{source.label} to %{target.label}: %{value:.1f} million passengers a year<extra></extra>")))
fig.update_layout(margin=dict(l=20, r=20, b=20))
flying = lambda g: P["pax"][g].fillna(0).iloc[-1] > 0
oc = pd.Series({o: gout[[g for g in gout.index if STATUS[g] == o]].sum() for o in OUTC if any(STATUS[g] == o for g in gout.index)}) / gout.sum() * 100
stopped_sh = gout[[g for g in gout.index if not flying(g)]].sum() / gout.sum() * 100
put("sankey", 3, "Where the pre-COVID passengers ended up", fig, 560,
    how="Left: the brands flying before COVID (width = average passengers a year, April 2017 to March 2020). Middle: the airline group each brand belongs to today. Right: what happened to that group since.",
    tells="What share of the pre-COVID market sits with airlines that recovered, failed or disappeared.",
    obs=[f"{oc.get('Recovered and holding', 0):.0f}% of pre-COVID passengers flew with IndiGo or the brands that now form Air India Group, the only two groups above their pre-COVID level today.",
         f"{stopped_sh:.0f}% flew with airlines that no longer exist (Jet Airways, JetLite, Go First and smaller regionals)."])

allg = [g for g in P["pax"].columns if (P["pax"][g] > 0).any()]
stages = [("Flew scheduled domestic flights since Jan 2017", allg), ("Flew before COVID (Apr 2017 to Mar 2020)", [g for g in allg if (P["pax"].loc[PRE, g] > 0).any()]),
          ("Flew every pre-COVID month, with at least 0.5% of passengers", BASE), ("Still flying today", [g for g in BASE if flying(g)]),
          ("Got back to the pre-COVID passenger level at some point", [g for g in BASE if flying(g) and STATUS[g].startswith("Recovered")]),
          ("Still above it today", [g for g in BASE if STATUS[g] == "Recovered and holding"])]
fills_ = ["#e0e0e0", "#d4e4f2", "#c1d9ee", "#a9cbe7", "#92bde0", "#7aafd9"]
fig = go.Figure()
for i, (d, v) in enumerate(stages):      # centred bars on a numeric axis, so each description sits exactly on its own box
    fig.add_trace(go.Bar(y=[i], x=[len(v)], base=[-len(v) / 2], orientation="h", width=0.82, marker=dict(color=fills_[i], line=dict(color="white", width=2)), showlegend=False,
                         customdata=[", ".join(sorted(v))], hovertemplate="%{customdata}<extra></extra>"))
    wide = len(v) >= 12      # wide boxes hold their text; narrow ones get it just to the right, on the same row
    fig.add_annotation(x=0 if wide else len(v) / 2 + 0.3, xanchor="center" if wide else "left", y=i, align="center" if wide else "left",
                       text=f"<b>{d}</b><br>{len(v)} airline group{'s' if len(v) != 1 else ''}", showarrow=False, font=dict(size=12, color="#222"))
fig.update_yaxes(visible=False, autorange="reversed"); fig.update_xaxes(visible=False, range=[-len(allg) / 2 - 0.5, len(allg) / 2 + 0.5]); fig.update_layout(margin=dict(l=20, r=20, b=20), bargap=0.1)
put("funnel", 3, "Which airlines made it through each filter?", fig, 500, "Hover over a box to see the airlines in it",
    how="Each box keeps only the airline groups that passed the filter above it; box width = number of groups.",
    tells="How the 20 airline groups in the data narrow down to the ones that recovered.",
    obs=[f"{len(stages[1][1])} of {len(allg)} groups flew before COVID, but only {len(BASE)} flew every pre-COVID month with a meaningful share: most of the others were small regional airlines that came and went.",
         f"Of those {len(BASE)}, {len(stages[3][1])} still fly, {len(stages[4][1])} got back to their pre-COVID level at some point and {len(stages[5][1])} ({', '.join(stages[5][1])}) are above it today."])


## 9. Operations, momentum and market share

In [9]:
# ---------------- SECTION 4: OPERATIONS ----------------
since = months >= "2023-06-01"
LFm = lf_of(P["rpk"], P["ask"]); TRIP = P["rpk"] * 1000 / P["pax"]
fig = go.Figure([go.Violin(y=LFm.loc[since, g].dropna(), name=g, line=dict(color=gc(g)), fillcolor=gc(g), opacity=.55, box_visible=True, meanline_visible=True,
                           points="all", pointpos=0, jitter=.3, marker=dict(size=3), spanmode="hard") for g in ORDER])
fig.update_yaxes(title="Monthly load factor (% of seat-km filled)"); fig.update_layout(showlegend=False)
sd_lf = LFm.loc[since, ORDER].std().sort_values(); medlf = LFm.loc[since, ORDER].median()
put("violin", 4, "How steady is the load factor?", fig, 480,
    how="Each dot is one month's load factor since mid-2023. The outline is a smoothed histogram of those months turned on its side and mirrored, so it is wide where many months fall. The box inside spans the middle half of the months with the median as a line.",
    tells="Not just how full an airline's planes are, but how reliably: a short, wide shape is steady; a tall, thin one swings.",
    obs=[f"{sd_lf.index[0]} has the fullest and steadiest planes (median {medlf[sd_lf.index[0]]:.0f}%, varying by {sd_lf.iloc[0]:.1f} points); {sd_lf.index[-1]} swings the most ({sd_lf.iloc[-1]:.1f} points), from about {LFm.loc[since, sd_lf.index[-1]].min():.0f}% to {LFm.loc[since, sd_lf.index[-1]].max():.0f}%.",
         f"SpiceJet's median ({medlf['SpiceJet']:.0f}%) is the {['highest','second-highest','third-highest'][list(medlf.sort_values(ascending=False).index).index('SpiceJet')]}: the few flights it still runs are full, which supports the view that it lost size, not passengers per flight."])

trip = {g: P["rpk"].loc[W12, g].sum() * 1000 / P["pax"].loc[W12, g].sum() for g in ORDER}
lf12 = {g: lf_of(P["rpk"].loc[W12, g].sum(), P["ask"].loc[W12, g].sum()) for g in ORDER}
askb = np.array([bn(P["ask"].loc[W12, g].sum()) for g in ORDER])
fig = go.Figure(go.Scatter(x=[trip[g] for g in ORDER], y=M0.loc[ORDER, "ppd"], mode="markers+text", text=ORDER, textposition=["bottom center" if g == "SpiceJet" else "top center" for g in ORDER],
        marker=dict(size=np.sqrt(askb) * 5 + 10, color=[gc(g) for g in ORDER], line=dict(color="#222", width=1)),
        customdata=[[lf12[g], PIL0.loc[g, "Operations"]] for g in ORDER], hovertemplate="%{text}<br>average trip %{x:.0f} km<br>%{y:.0f} passengers per departure<br>load factor %{customdata[0]:.0f}%<br>Operations score %{customdata[1]:.0f}<extra></extra>"))
fig.add_shape(type="rect", x0=350, x1=650, y0=30, y1=70, line=dict(dash="dot", color="#999")); fig.add_annotation(x=500, y=73, text="Regional model: small aircraft, short hops", showarrow=False, font=dict(size=11, color="#555"))
fig.add_shape(type="rect", x0=900, x1=1250, y0=130, y1=185, line=dict(dash="dot", color="#999")); fig.add_annotation(x=1000, y=191, text="Jet model: big aircraft, longer trips", showarrow=False, font=dict(size=11, color="#555"))
fig.update_xaxes(title="Average trip length (km)"); fig.update_yaxes(title="Passengers per departure", range=[20, 200])
jets = [g for g in ORDER if trip[g] > 800]; regs = [g for g in ORDER if trip[g] <= 800]
put("trip", 4, "Two business models", fig, 480, "Bubble size = seats offered; hover for load factor and Operations score",
    how="Each bubble is an airline over the last 12 months: across = average trip length (passenger-km / passengers), up = passengers per departure.",
    tells="The airlines split into two business models. Because two of the three Operations metrics reward big aircraft and long trips, this chart explains most of the Operations pillar.",
    obs=[f"Jet airlines carry about {M0.loc[jets, 'ppd'].mean():.0f} passengers per flight over about {np.mean([trip[g] for g in jets]):.0f} km; regional airlines about {M0.loc[regs, 'ppd'].mean():.0f} over {np.mean([trip[g] for g in regs]):.0f} km.",
         f"So a low Operations score for {', '.join(regs)} reflects their aircraft and routes, not poor management; compare them with each other, not with IndiGo. Akasa Air flies the longest average trip ({trip['Akasa Air']:.0f} km)."])

dd = pd.DataFrame({"trip": TRIP.loc[since, ORDER].stack(), "lf": LFm.loc[since, ORDER].stack()}).dropna(); dd.index.names = ["Date", "g"]; dd = dd.reset_index()
fig = go.Figure()
for g in ORDER:
    d_ = dd[dd.g == g]
    fig.add_trace(go.Histogram2dContour(x=d_.trip, y=d_.lf, name=g, legendgroup=g, showlegend=True, ncontours=5, contours=dict(coloring="lines"), colorscale=[[0, gc(g)], [1, gc(g)]], showscale=False, line=dict(width=1.5), hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=d_.trip, y=d_.lf, mode="markers", name=g, legendgroup=g, showlegend=False, marker=dict(color=gc(g), symbol=gs(g), size=5, opacity=.6),
                             text=d_.Date.dt.strftime("%b %Y"), hovertemplate=g + "<br>%{text}: trip %{x:.0f} km, load factor %{y:.1f}%<extra></extra>"))
fig.update_xaxes(title="Average trip length (km)"); fig.update_yaxes(title="Monthly load factor (%)")
zs = dd.groupby("g").agg(ts=("trip", "std"), ls=("lf", "std"))
within = np.nanmedian([spearman(dd[dd.g == g].trip, dd[dd.g == g].lf) for g in ORDER])
put("contour", 11, "Each airline's operating zone", fig, 500, "Click names in the legend to show or hide airlines",
    how="Each dot is one airline-month since mid-2023 (trip length across, load factor up). The contour lines in the airline's colour enclose where its months cluster, like elevation lines on a map.",
    tells="Each airline's typical operating point and how much it moves around it.",
    obs=[f"Across airlines, longer trips go with fuller planes (rank correlation {spearman(dd.trip, dd.lf):+.2f}), but within one airline the link disappears ({within:+.2f}): each airline stays in its own zone, set by its network.",
         f"IndiGo's zone is the tightest (its months vary by only {zs.loc['IndiGo','ts']:.0f} km in trip length and {zs.loc['IndiGo','ls']:.1f} points in load factor). {zs.ls.idxmax()} swings most on load factor ({zs.ls.max():.1f} points); {zs.ts.idxmax()} moves most along trip length ({zs.ts.max():.0f} km) as it added longer routes."])

br = sd[sd.Date.isin(W12)].groupby(["Group", "Airline"])[["Passenger Kilometers"]].sum().reset_index(); br = br[(br["Passenger Kilometers"] > 0) & br.Group.isin(ORDER)]
ids, labels, parents, values, cols, hv = [], [], [], [], [], []
for g in ORDER:
    d_ = br[br.Group == g]; ids.append(g); labels.append(g); parents.append(""); values.append(bn(d_["Passenger Kilometers"].sum())); cols.append(gc(g)); hv.append(f"load factor {lf12[g]:.0f}%")
    if d_.Airline.nunique() > 1:
        for a_ in d_.Airline:
            m_ = sd[sd.Date.isin(W12) & (sd.Airline == a_)]
            ids.append(f"{g}/{a_}"); labels.append(a_); parents.append(g); values.append(bn(d_.loc[d_.Airline == a_, "Passenger Kilometers"].sum())); cols.append(tint(gc(g)))
            hv.append(f"load factor {lf_of(m_['Passenger Kilometers'].sum(), m_['Seat Kilometers'].sum()):.0f}%")
fig = go.Figure(go.Sunburst(ids=ids, labels=labels, parents=parents, values=values, branchvalues="total", marker=dict(colors=cols, line=dict(color="white", width=2)), textfont=dict(color="#222"),
                            customdata=hv, hovertemplate="<b>%{label}</b><br>%{value:.1f} bn passenger-km (%{percentRoot:.0%})<br>%{customdata}<extra></extra>", insidetextorientation="radial"))
fig.update_layout(margin=dict(l=10, r=10, b=10))
sh12 = pd.Series({g: P["rpk"].loc[W12, g].sum() for g in ORDER}); sh12 = sh12 / sh12.sum() * 100
aig = br[br.Group == "Air India Group"].set_index("Airline")["Passenger Kilometers"]; aig = aig / aig.sum() * 100
put("sunburst", 11, "Who flies the passenger-km, brand by brand", fig, 520, "Hover for passenger-km, share and load factor",
    how="Inner ring = airline groups, outer ring = the brands inside a group (lighter shade of the group's colour). Angle = passenger-km over the last 12 months.",
    tells="How concentrated the market is and how the Air India Group is split after its mergers.",
    obs=[f"IndiGo and Air India Group together fly {sh12['IndiGo'] + sh12['Air India Group']:.0f}% of all passenger-km.",
         "Inside Air India Group: " + ", ".join(f"{a} {v:.0f}%" for a, v in aig.sort_values(ascending=False).items()) + " (Vistara and AirAsia India no longer appear as they have been merged in)."])

# ---------------- SECTION 5: MOMENTUM AND MARKET SHARE ----------------
grpA = {"IndiGo": P["ask"]["IndiGo"], "Air India Group": P["ask"]["Air India Group"], "All other airlines": P["ask"].drop(columns=["IndiGo", "Air India Group"]).sum(axis=1)}
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=list(grpA), vertical_spacing=0.08); qs = {}
for j, (nm, s) in enumerate(grpA.items(), 1):
    s = s.loc["2021-12-01":] / 1e6; q = s.loc["2022-01-01":].groupby(s.loc["2022-01-01":].index.to_period("Q"))
    o = s.groupby(s.index.to_period("Q")).last().shift(1).reindex(q.last().index); c = q.last()
    h = pd.concat([q.max(), o], axis=1).max(axis=1); l = pd.concat([q.min(), o], axis=1).min(axis=1); qs[nm] = (o, c)
    fig.add_trace(go.Candlestick(x=c.index.to_timestamp(), open=o, high=h, low=l, close=c, name=nm, showlegend=False,
                                 increasing=dict(line=dict(color="#2166AC"), fillcolor="#2166AC"), decreasing=dict(line=dict(color="#B35806"), fillcolor="#FFFFFF")), row=j, col=1)
    fig.update_yaxes(title_text="bn seat-km", row=j, col=1)
fig.update_xaxes(rangeslider_visible=False); fig.update_annotations(font_size=12)
up = {nm: int((c > o).sum()) for nm, (o, c) in qs.items()}; grow = {nm: (c.iloc[-1] / o.iloc[0] - 1) * 100 for nm, (o, c) in qs.items()}
put("candle", 5, "Capacity, quarter by quarter", fig, 660, "Seat-km offered per month; filled blue = grew over the quarter, hollow orange = shrank",
    how="One candle per quarter. It opens at the last month of the previous quarter and closes at the last month of this one, so the candles join up; the wicks reach the highest and lowest month.",
    tells="Who is adding capacity (the supply side of recovery) and how steadily.",
    obs=[f"Since the start of 2022 IndiGo's monthly capacity grew {grow['IndiGo']:+.0f}% and Air India Group's {grow['Air India Group']:+.0f}%, while all other airlines together changed {grow['All other airlines']:+.0f}%.",
         f"All other airlines added capacity in only {up['All other airlines']} of {len(qs['IndiGo'][0])} quarters; the big orange candle in 2023 is Go First stopping."])

tmo = months[months >= "2019-01-01"]
three = pd.DataFrame({"IndiGo": P["rpk"]["IndiGo"], "Air India Group": P["rpk"]["Air India Group"], "All others": P["rpk"].drop(columns=["IndiGo", "Air India Group"]).sum(axis=1)}).loc[tmo].fillna(0) / 1e6
RMAX = float(np.sqrt(three.values.max())) * 1.05
def sector(T):
    v = three.loc[T]; return go.Barpolar(r=np.sqrt(v.values), theta=[60, 180, 300], width=[118] * 3, marker=dict(color=[gc("IndiGo"), gc("Air India Group"), "#9E9E9E"], line=dict(color="white", width=2)),
                                         customdata=np.c_[v.values, v.values / max(v.sum(), 1e-9) * 100], text=list(three.columns), hovertemplate="%{text}: %{customdata[0]:.1f} bn passenger-km (%{customdata[1]:.0f}%)<extra></extra>")
fig = go.Figure([sector(tmo[0])], frames=[go.Frame(name=f"{T:%Y-%m}", data=[sector(T)], layout=dict(annotations=[dict(text=f"<b>{T:%B %Y}</b>", x=0.5, y=1.02, xref="paper", yref="paper", showarrow=False, font=dict(size=15))])) for T in tmo])
fig.update_layout(polar=dict(radialaxis=dict(range=[0, RMAX], showticklabels=False, ticks=""), angularaxis=dict(rotation=90, direction="clockwise", tickvals=[60, 180, 300], ticktext=list(three.columns))),
                  annotations=[dict(text=f"<b>{tmo[0]:%B %Y}</b>", x=0.5, y=1.02, xref="paper", yref="paper", showarrow=False, font=dict(size=15))], showlegend=False, margin=dict(b=90),
                  updatemenus=[dict(type="buttons", x=0.0, y=-0.08, xanchor="left", direction="right", buttons=[dict(label="▶ Play", method="animate", args=[None, dict(frame=dict(duration=140, redraw=True), transition=dict(duration=0), fromcurrent=True)]),
                               dict(label="⏸ Pause", method="animate", args=[[None], dict(mode="immediate", frame=dict(duration=0, redraw=False))])])],
                  sliders=[dict(x=0.2, y=-0.04, len=0.8, currentvalue=dict(visible=False), steps=[dict(label=f"{T:%Y}" if T.month == 1 else "", method="animate", args=[[f"{T:%Y-%m}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for T in tmo])])
s19, s12 = three.loc["2019"].sum() / three.loc["2019"].sum().sum() * 100, three.loc[W12].sum() / three.loc[W12].sum().sum() * 100
put("sectors", 5, "The three-way split, month by month", fig, 580, "Press Play or drag the slider",
    how="Three 120-degree sectors, one each for IndiGo, Air India Group and all other airlines. A sector's area is proportional to its passenger-km in that month, so the whole shape shrinks in the lockdown and grows after.",
    tells="How the balance of the market shifted month by month, and how big the market was at each point.",
    obs=[f"IndiGo's share of passenger-km went from {s19['IndiGo']:.0f}% in 2019 to {s12['IndiGo']:.0f}% in the last 12 months; Air India Group from {s19['Air India Group']:.0f}% to {s12['Air India Group']:.0f}%.",
         f"The grey 'all others' sector shrinks from {s19['All others']:.0f}% to {s12['All others']:.0f}%: the market has become a two-airline market."])

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.17, subplot_titles=["Size: share of passenger-km (%)", "Performance: index (0 to 100)"])
fig.add_trace(go.Bar(y=ORDER, x=[sh12[g] for g in ORDER], orientation="h", marker=dict(color=[gc(g) for g in ORDER]), text=[(f"{sh12[g]:.0f}%" if sh12[g] >= 1 else f"{sh12[g]:.1f}%") for g in ORDER],
                     textposition="outside", cliponaxis=False, showlegend=False, hovertemplate="%{y}: %{x:.1f}% of passenger-km<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(y=ORDER, x=PIL0.loc[ORDER, "Index"], orientation="h", marker=dict(color=[gc(g) for g in ORDER]), text=[f"{v:.0f}" for v in PIL0.loc[ORDER, "Index"]], textposition="outside",
                     cliponaxis=False, showlegend=False, hovertemplate="%{y}: index %{x:.0f}<extra></extra>"), row=1, col=2)
fig.update_xaxes(autorange="reversed", range=[0, 75], row=1, col=1, showgrid=False); fig.update_xaxes(range=[0, 100], row=1, col=2, showgrid=False)
fig.add_vline(x=50, line=dict(dash="dot", color="#888"), row=1, col=2)
fig.update_yaxes(autorange="reversed", side="right", tickfont=dict(size=12), row=1, col=1); fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=2); fig.update_layout(margin=dict(l=40, r=40))
put("sizeperf", 5, "Is size the same as performance?", fig, 420,
    how="Each airline is one row in its own colour; the left bar is its size (share of passenger-km), the right bar its performance index (dotted line = 50, typical).",
    tells="Whether being big and performing well go together.",
    obs=[f"Akasa Air is a tenth of IndiGo's size ({sh12['Akasa Air']:.0f}% vs {sh12['IndiGo']:.0f}%) yet scores almost as high ({PIL0.loc['Akasa Air', 'Index']:.0f} vs {PIL0.loc['IndiGo', 'Index']:.0f}).",
         f"SpiceJet is the fourth-largest airline but scores below the typical 50 ({PIL0.loc['SpiceJet', 'Index']:.0f}): size alone is not performance."])

CAL = pd.DataFrame({"y": months.year, "m": months.month, "v": mk_rpk.values / 1e6}).pivot(index="y", columns="m", values="v").reindex(columns=range(1, 13))
MONTHS = [calendar.month_abbr[m] for m in range(1, 13)]
fig = go.Figure(go.Heatmap(z=CAL.values, x=MONTHS, y=[str(y) for y in CAL.index], colorscale=[[0, "#fff7ec"], [0.35, "#fdd49e"], [0.7, "#fc8d59"], [1, "#b30000"]], xgap=2, ygap=2,
                           colorbar=dict(title=dict(text="bn pax-km")), hovertemplate="%{x} %{y}: %{z:.1f} bn passenger-km<extra></extra>"))
fig.update_yaxes(autorange="reversed", type="category")
aft = CAL.loc[2023:]; above19 = int((aft.sub(CAL.loc[2019], axis=1) > 0).sum().sum()); n_after = int(aft.notna().sum().sum())
recent = CAL.loc[[y for y in CAL.index if y >= LAST.year - 1]].mean()
put("calendar", 5, "Which months are busy?", fig, 460, "Darker = more passenger-km; hover for values",
    how="A calendar: each row is a year, each column a month, and the colour is the whole market's passenger-km that month.",
    tells="The seasonal rhythm of air travel, how deep the COVID hole was, and how each year compares.",
    obs=[f"Every year shows the same rhythm: busiest in {' and '.join(MONTHS[m - 1] for m in recent.nlargest(2).index)}, quietest in {' and '.join(MONTHS[m - 1] for m in recent.nsmallest(2).index)} (the monsoon). The pale block from April 2020 is the lockdown; the second pale patch in April-June 2021 is the second wave.",
         f"From 2023 on, {above19} of {n_after} months beat the same month of 2019, so the market is bigger in every season. But July and August {LAST.year} ({CAL.loc[LAST.year, 7]:.1f} and {CAL.loc[LAST.year, 8]:.1f} bn) came in below the same months of {LAST.year - 1} ({CAL.loc[LAST.year - 1, 7]:.1f} and {CAL.loc[LAST.year - 1, 8]:.1f} bn), a sign that growth has paused this year."])

hist_ = mk_rpk.loc[LAST - pd.DateOffset(months=35):] / 1e6
back_x = list(months[(months >= LAST - pd.DateOffset(months=23))]); back = [forecast(mk_rpk, T - pd.DateOffset(months=1), 1).iloc[0] / 1e6 for T in back_x]
bb1 = np.array(back) * MAPE[1] / 100
fc = forecast(mk_rpk, LAST); fx = pd.date_range(LAST + pd.DateOffset(months=1), periods=6, freq="MS"); band = fc.values * (MAPE.values / 100) / 1e6
px_ = [LAST] + list(fx); py = [mk_rpk.loc[LAST] / 1e6] + list(fc.values / 1e6); up_ = [py[0]] + list(fc.values / 1e6 + band); lo_ = [py[0]] + list(fc.values / 1e6 - band)
fig = go.Figure([go.Scatter(x=back_x + back_x[::-1], y=list(np.array(back) + bb1) + list((np.array(back) - bb1)[::-1]), fill="toself", mode="lines", fillcolor="rgba(33,102,172,.15)", line=dict(width=0), name="Band (past)", hoverinfo="skip"),
                 go.Scatter(x=px_ + px_[::-1], y=up_ + lo_[::-1], fill="toself", mode="lines", fillcolor="rgba(203,24,29,.15)", line=dict(width=0), name="Band (future)", hoverinfo="skip"),
                 go.Scatter(x=hist_.index, y=hist_, mode="lines", name="Actual", line=dict(color="#111", width=2.5)),
                 go.Scatter(x=back_x, y=back, mode="lines", name="Predicted (one month ahead)", line=dict(color="#2166AC", width=2, dash="dash")),
                 go.Scatter(x=px_, y=py, mode="lines+markers", name="Projection", line=dict(color="#CB181D", width=2.5, dash="dash"), marker=dict(size=7))])
fig.update_yaxes(title="Passenger-km per month (billions)", rangemode="tozero"); fig.update_xaxes(title="Month")
miss = (np.array(back) - (mk_rpk.loc[back_x].values / 1e6)); worst_m = back_x[int(np.argmax(np.abs(miss)))]
put("fan", 5, "Predicted vs actual, and the next six months", fig, 480, "Blue = what the rule would have predicted for past months; red = projection",
    how="A simple rule: next month = the same month last year x the latest 12-month growth rate. Blue dashed shows what the rule would have predicted for each past month, red dashed its projection; the bands are its typical past error.",
    tells="How predictable the market is from its own seasonality and trend, and a rough six-month outlook.",
    obs=[f"The rule misses by {MAPE[1]:.1f}% on average one month ahead and {MAPE[6]:.1f}% six months ahead; its biggest miss was {worst_m:%B %Y}.",
         f"It projects {fc.iloc[0] / 1e6:.1f} billion passenger-km for {fx[0]:%B %Y} and {fc.max() / 1e6:.1f} billion at the winter peak. It is a rule of thumb, not a forecast model: it cannot foresee shocks such as an airline collapse."])


## 10. Route network and international flights

In [10]:
# ---------------- SECTION 6: THE ROUTE NETWORK (city-pair data; routes are compared with 2019, the last full year before COVID) ----------------
r19c = CR[CR.Date.dt.year == 2019].groupby(["p1", "p2"]).pax.sum(); rwc = CR[CR.Date.isin(WC)].groupby(["p1", "p2"]).pax.sum()
RT = pd.concat([r19c.rename("p19"), rwc.rename("now")], axis=1).fillna(0); T19c, TNc = RT.p19.sum(), RT.now.sum()
RT["route"] = [f"{a.title()} - {b.title()}" for a, b in RT.index]
cl = lambda c: c.title()

x17, x18 = XCHK[XCHK.index.year == 2017], XCHK[XCHK.index.year >= 2018]
fig = go.Figure()
fig.add_hrect(y0=0.98, y1=1.02, fillcolor="#e8eef5", line_width=0, layer="below")
fig.add_trace(go.Scatter(x=XCHK.index, y=XCHK.values, mode="lines+markers", marker=dict(size=4, color="#2166AC"), line=dict(color="#2166AC", width=1.5), showlegend=False, hovertemplate="%{x|%b %Y}: %{y:.3f}<extra></extra>"))
fig.add_hline(y=1, line=dict(color="#222", dash="dash")); fig.update_yaxes(title="City-pair / airline passengers", range=[0.9, 1.1]); fig.update_xaxes(title="Month")
put("xchk", 6, "Do the two sources agree?", fig, 360, "Shaded band = within 2%",
    how="For each month: total passengers in the city-pair file divided by total passengers in the airline file. 1.00 means the two DGCA publications agree exactly.",
    tells="Whether the second data source can be trusted before we use it to study routes.",
    obs=[f"From 2018 they agree almost exactly (median ratio {x18.median():.3f}), so the route analysis below rests on the same passengers as the airline analysis.",
         f"The 2017 gap ({(x17.min() - 1) * 100:.0f}% to {(x17.max() - 1) * 100:.0f}%) is AirAsia India's 2017 airline file, which DGCA's server refused to send (HTTP 403); this is why route comparisons use 2019 as the base year."])

mkc = CR.groupby(["Date", "kind"]).pax.sum().unstack(); bb = mkc[mkc.index.year == 2019]; bb.index = bb.index.month
rat = (mkc / bb.reindex(mkc.index.month).values * 100).rolling(3).mean().loc["2019-03-01":]
KC = {"Metro to metro": "#B35806", "Metro to other city": "#2166AC", "Other city to other city": "#009E73"}; KD = {"Metro to metro": "solid", "Metro to other city": "dash", "Other city to other city": "dot"}
fig = go.Figure([go.Scatter(x=rat.index, y=rat[k], name=k, mode="lines", line=dict(color=KC[k], width=2.5, dash=KD[k])) for k in KC])
fig.add_hline(y=100, line=dict(color="#222", width=1)); fig.update_yaxes(title="% of the same months in 2019"); fig.update_xaxes(title="Month")
kk = RT.join(CR.drop_duplicates(["p1", "p2"]).set_index(["p1", "p2"])["kind"]).groupby("kind")[["p19", "now"]].sum(); pct_k = kk.now / kk.p19 * 100
fb_k = {k: rat.loc["2020-06-01":, k][lambda s: s >= 100].index.min() for k in KC}
put("routetype", 6, "Which kinds of routes came back first?", fig, 440, f"Metros: {', '.join(m.title() for m in METRO)}. 3-month averages.",
    how="Every route is put in one of three groups by its two ends. Each line is that group's passengers as a % of the same months of 2019.",
    tells="Whether the recovery was led by the big trunk routes or by smaller cities.",
    obs=[f"Routes between two non-metro cities recovered first and furthest: {pct_k['Other city to other city']:.0f}% of 2019 now, against {pct_k['Metro to metro']:.0f}% for metro-to-metro routes.",
         "First month back at 2019 level: " + "; ".join(f"{k.lower()} {d:%b %Y}" if pd.notna(d) else f"{k.lower()} not yet" for k, d in fb_k.items()) + ". The growth came from smaller cities joining the network (new routes such as Darbhanga and Ayodhya), not from the old trunk routes."])

CT = pd.concat([CR.rename(columns={"p1": "city"})[["Date", "city", "pax"]], CR.rename(columns={"p2": "city"})[["Date", "city", "pax"]]])
CT["city2"] = np.where(CT.city.isin(METRO), CT.city.str.title(), "All other cities")
cy = CT.groupby(["city2", "Date"]).pax.sum().unstack(0)
yrs = list(range(2019, LASTC.year + 1)); rows = {}
for y in yrs:
    mm = cy.index[cy.index.year == y]; base = cy[cy.index.year == 2019]; base = base[base.index.month.isin(mm.month)]
    rows[y] = cy.loc[mm].sum() / base.sum() * 100
HM = pd.DataFrame(rows).loc[[m.title() for m in METRO] + ["All other cities"]]
lbl = [str(y) if y < LASTC.year else f"{y} (Jan-{LASTC:%b})" for y in yrs]
fig = go.Figure(go.Heatmap(z=HM.values, x=lbl, y=HM.index, colorscale=DIV, zmid=100, zmin=40, zmax=160, text=HM.round(0).values, texttemplate="%{text:.0f}", xgap=2, ygap=2,
                           colorbar=dict(title=dict(text="% of 2019")), hovertemplate="%{y}, %{x}: %{z:.0f}% of 2019<extra></extra>"))
fig.update_yaxes(autorange="reversed"); last_col = HM.iloc[:, -1]; bestm = last_col.drop("All other cities").idxmax(); below_m = last_col.drop("All other cities")[lambda s: s < 100]
put("metroheat", 6, "Traffic at the six metros, year by year", fig, 400, "Blue = above 2019, orange = below",
    how="Each cell is the passengers on all routes touching that city in that year, as a % of the same months of 2019 (so the part-year column is fair).",
    tells="Which big airports have grown past 2019 and which are lagging.",
    obs=[f"{bestm} leads the metros ({last_col[bestm]:.0f}% of 2019 this year); 'all other cities' together are at {last_col['All other cities']:.0f}%, ahead of {int((last_col.drop('All other cities') < last_col['All other cities']).sum())} of the 6 metros.",
         (f"{' and '.join(f'{c} ({v:.0f}%)' for c, v in below_m.items())} {'is' if len(below_m) == 1 else 'are'} still below 2019." if len(below_m) else "Every metro is now above 2019.") + " 2021 is the weakest column everywhere: the second wave."])

topR = RT.sort_values("p19", ascending=False).head(12); upr = topR.now >= topR.p19
fig = go.Figure()
for r in topR.itertuples():
    fig.add_trace(go.Scatter(x=[r.p19 / 1e6, r.now / 1e6], y=[r.route, r.route], mode="lines", line=dict(color="#c8c8c8", width=5), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=topR.p19 / 1e6, y=topR.route, mode="markers", name="2019", marker=dict(symbol="circle-open", size=12, color="#333", line=dict(width=2.5)), hovertemplate="%{y}: %{x:.2f} M in 2019<extra></extra>"))
for flag, nm, col in [(True, "Last 12 months: above 2019", "#2166AC"), (False, "Last 12 months: below 2019", "#B35806")]:
    d_ = topR[upr == flag]
    fig.add_trace(go.Scatter(x=d_.now / 1e6, y=d_.route, mode="markers", name=nm, marker=dict(symbol="diamond", size=12, color=col), hovertemplate="%{y}: %{x:.2f} M now<extra></extra>"))
fig.add_trace(go.Scatter(x=topR[["p19", "now"]].max(axis=1) / 1e6 + 0.12, y=topR.route, mode="text", text=[f"{v:.0f}%" for v in topR.now / topR.p19 * 100], textposition="middle right", showlegend=False, hoverinfo="skip"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title="Passengers in 12 months (millions, both directions)"); fig.update_layout(margin=dict(l=170))
tp = (topR.now / topR.p19 * 100)
put("toproutes", 6, "Did the dominant routes grow?", fig, 520, "The 12 busiest routes of 2019; labels = last 12 months as % of 2019",
    how="Circle = a route's passengers in 2019, diamond = last 12 months; the grey bar is the change. Blue diamonds are above 2019, orange below.",
    tells="Whether the trunk routes that carried the pre-COVID market are back.",
    obs=[f"The busiest routes are roughly flat: all 12 sit between {tp.min():.0f}% and {tp.max():.0f}% of 2019 ({int(upr.sum())} above, {12 - int(upr.sum())} below), while all routes together are at {TNc / T19c * 100:.0f}%. Delhi - Mumbai, the busiest route in India, is at {tp.iloc[0]:.0f}%.",
         f"The biggest gain is {tp.idxmax() if isinstance(tp.idxmax(), str) else topR.route[tp.idxmax()]} ({tp.max():.0f}%), the weakest {topR.route[tp.idxmin()]} ({tp.min():.0f}%): the market grew around the old trunk routes rather than through them."])

lost = RT[(RT.p19 > 0) & (RT.now == 0)]; new = RT[(RT.p19 == 0) & (RT.now > 0)]; kept = RT[(RT.p19 > 0) & (RT.now > 0)]
grow_k = (kept.now.sum() - kept.p19.sum()) / 1e6
fig = go.Figure([go.Bar(y=["2019", "Last 12 months"], x=[kept.p19.sum() / 1e6, kept.now.sum() / 1e6], orientation="h", name=f"Routes flown in both periods ({len(kept)})", marker=dict(color="#c6dbef", line=dict(color="#6baed6", width=1)),
                        text=[f"{kept.p19.sum() / 1e6:.0f} M", f"{kept.now.sum() / 1e6:.0f} M"], textposition="inside", insidetextanchor="middle", textfont=dict(color="#08306b", size=14)),
                 go.Bar(y=["2019"], x=[lost.p19.sum() / 1e6], orientation="h", name=f"Routes dropped since 2019 ({len(lost)})", marker=dict(color="#CB181D"), text=[f"{lost.p19.sum() / 1e6:.1f} M"], textposition="outside", textfont=dict(color="#CB181D", size=14)),
                 go.Bar(y=["Last 12 months"], x=[new.now.sum() / 1e6], orientation="h", name=f"New routes since 2019 ({len(new)})", marker=dict(color="#08519c"), text=[f"{new.now.sum() / 1e6:.1f} M"], textposition="outside", textfont=dict(color="#08519c", size=14))])
fig.add_annotation(x=kept.now.sum() / 1e6 / 2, y=0.5, text=f"same routes: {grow_k:+.0f} M passengers", showarrow=False, font=dict(size=12, color="#08306b"))
fig.update_layout(barmode="stack", margin=dict(l=110), legend=dict(orientation="h", y=-0.3, x=0)); fig.update_xaxes(title="Passengers in 12 months (millions)"); fig.update_yaxes(autorange="reversed")
nb_ = new.sort_values("now", ascending=False).head(3)
put("churn", 6, "Where did the extra passengers come from?", fig, 340, "Light blue = routes flown in both periods; red = routes dropped; dark blue = new routes",
    how="Every route is sorted into three groups: flown in both 2019 and the last 12 months, flown only in 2019 (dropped, red) and flown only now (new, dark blue). The top bar is 2019, the bottom bar the last 12 months.",
    tells="Whether the market grew by flying the same routes more, or by opening new ones.",
    obs=[f"Most of the growth is on routes that already existed: they gained {grow_k:.0f} million passengers. New routes add {new.now.sum() / 1e6:.1f} million, dropped routes lost {lost.p19.sum() / 1e6:.1f} million.",
         f"{len(new)} routes are new since 2019 (largest: {', '.join(nb_.route)}) and {len(lost)} were dropped; the network churns at the edges while the core carries the growth."])

kb = kept[kept.p19 >= 100_000].copy(); kb["pct"] = kb.now / kb.p19 * 100; kb["bin"] = (kb.pct.clip(upper=309) // 10 * 10).astype(int)
LO, HI = 50, 200
def tail_txt(d):
    d = d.sort_values("pct"); s_ = [f"{r} ({p:.0f}%)" for r, p in zip(d.route, d.pct)]
    return "<br>".join(s_[:14] + ([f"... and {len(s_) - 14} more"] if len(s_) > 14 else []))
bins = pd.DataFrame({"b": range(0, 310, 10)}); cnt = kb.groupby("bin").size(); bins["n"] = bins.b.map(cnt).fillna(0).astype(int)
bins["zone"] = np.where(bins.b < LO, "low", np.where(bins.b >= HI, "high", "mid"))
bins["hv"] = [tail_txt(kb[kb.bin == b]) if z != "mid" and n else "" for b, z, n in zip(bins.b, bins.zone, bins.n)]
ZC = {"low": "#B35806", "mid": "#9ecae1", "high": "#08519c"}; ZN = {"low": f"Below {LO}% of 2019 (hover for routes)", "mid": "Middle of the distribution", "high": f"{HI}% of 2019 or more (hover for routes)"}
fig = go.Figure()
for z in ("low", "mid", "high"):
    d_ = bins[bins.zone == z]
    lab = [f"{b}-{b + 10}%" if b < 300 else "300%+" for b in d_.b]
    fig.add_trace(go.Bar(x=d_.b + 5, y=d_.n, width=9.4, name=ZN[z], marker=dict(color=ZC[z]), customdata=np.c_[lab, d_.hv],
                         hovertemplate=("<b>%{customdata[0]}: %{y} routes</b><br>%{customdata[1]}<extra></extra>" if z != "mid" else "%{customdata[0]}: %{y} routes<extra></extra>")))
fig.add_vline(x=100, line=dict(color="#222", dash="dash"), annotation_text="2019 level", annotation_position="top")
fig.update_xaxes(title="Passengers in the last 12 months, % of 2019 (last bar = 300% or more)", tickvals=list(range(0, 310, 50))); fig.update_yaxes(title="Number of routes")
fig.update_layout(bargap=0, hoverlabel=dict(align="left"), legend=dict(orientation="h", y=-0.25, x=0))
n_lo, n_hi = int((kb.pct < LO).sum()), int((kb.pct >= HI).sum())
put("routehist", 6, "How many routes are back above 2019?", fig, 440, "Routes with at least 100,000 passengers in 2019; hover the coloured end bars to see the routes",
    how="Each bar counts routes by how their last 12 months compare with 2019, in steps of 10%. The orange bars on the left (routes that lost more than half their traffic) and the dark blue bars on the right (routes that at least doubled) list their routes on hover.",
    tells="Whether recovery is spread across the network or driven by a few routes.",
    obs=[f"{(kb.pct >= 100).mean() * 100:.0f}% of the {len(kb)} routes are above 2019; the middle of the distribution sits around {kb.pct.median():.0f}%.",
         f"{n_lo} routes lost more than half their traffic (largest 2019 route among them: {kb[kb.pct < LO].sort_values('p19').route.iloc[-1] if n_lo else '-'}), and {n_hi} at least doubled (biggest jump: {kb.sort_values('pct').route.iloc[-1]}, {kb.pct.max():.0f}%)."])

def lorenz(v):
    v = np.sort(v[v > 0]); cx = np.r_[0, np.arange(1, len(v) + 1) / len(v)]; cy_ = np.r_[0, np.cumsum(v) / v.sum()]
    return cx, cy_, 1 - 2 * (np.trapezoid(cy_, cx) if hasattr(np, "trapezoid") else np.trapz(cy_, cx))
x1, y1, g19 = lorenz(RT.p19.values); x2, y2, gnw = lorenz(RT.now.values)
top10 = lambda v: np.sort(v[v > 0])[::-1][:max(1, int(len(v[v > 0]) * .1))].sum() / v.sum() * 100
fig = go.Figure([go.Scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color="#aaa", dash="dot"), name="Every route equal"),
                 go.Scatter(x=x1 * 100, y=y1 * 100, mode="lines", line=dict(color="#333", width=2.5), name=f"2019 (Gini {g19:.2f})"),
                 go.Scatter(x=x2 * 100, y=y2 * 100, mode="lines", line=dict(color="#2166AC", width=2.5, dash="dash"), name=f"Last 12 months (Gini {gnw:.2f})")])
fig.update_xaxes(title="Routes, smallest to largest (cumulative %)"); fig.update_yaxes(title="Passengers (cumulative %)")
put("lorenz", 6, "Is traffic concentrated on a few routes?", fig, 440,
    how="Routes are sorted from smallest to largest; the curve shows what share of passengers the smallest x% of routes carry. A curve sagging far below the diagonal means traffic sits on a few routes. The Gini number summarises the sag (0 = equal, 1 = one route has everything).",
    tells="Whether the network became more or less concentrated after COVID.",
    obs=[f"The busiest 10% of routes carry {top10(RT.now.values):.0f}% of passengers (2019: {top10(RT.p19.values):.0f}%); Gini {g19:.2f} then and {gnw:.2f} now.",
         "Route concentration hardly changed while airline concentration rose sharply (IndiGo and Air India Group now fly 90% of passenger-km): the same routes, fewer airlines flying them."])

cty = CT.assign(y19=CT.Date.dt.year == 2019, w=CT.Date.isin(WC)).groupby("city").apply(lambda d: pd.Series({"p19": d.loc[d.y19, "pax"].sum(), "now": d.loc[d.w, "pax"].sum()}))
cty = cty[cty.p19 >= 500_000]; cty["pct"] = cty.now / cty.p19 * 100 - 100; cty = cty.sort_values("pct")
belowC = cty[cty.pct < 0].sort_values("pct"); metC = cty.loc[[m for m in METRO if m in cty.index]].sort_values("pct")
gainC = cty[~cty.index.isin(METRO)].sort_values("pct").tail(10)
panels = [("Still below 2019", belowC), ("The six metros", metC), ("Ten fastest-growing cities", gainC)]
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{t} ({len(d)})" if i == 0 else t for i, (t, d) in enumerate(panels)], horizontal_spacing=0.13)
for j, (t, d) in enumerate(panels, 1):
    fig.add_trace(go.Bar(x=d.pct, y=[cl(c) for c in d.index], orientation="h", marker=dict(color=["#2166AC" if v >= 0 else "#B35806" for v in d.pct]),
                         text=[f"{v:+.0f}%" for v in d.pct], textposition="outside", cliponaxis=False, showlegend=False,
                         customdata=np.c_[d.p19 / 1e6, d.now / 1e6], hovertemplate="%{y}: %{x:+.0f}% vs 2019<br>2019 %{customdata[0]:.1f} M, now %{customdata[1]:.1f} M passengers<extra></extra>"), row=1, col=j)
    lo_, hi_ = min(d.pct.min(), 0), max(d.pct.max(), 0); padx = (hi_ - lo_) * 0.3 + 5
    fig.update_xaxes(range=[lo_ - (padx if lo_ < 0 else 0), hi_ + (padx if hi_ > 0 else 5)], zeroline=True, zerolinecolor="#222", row=1, col=j)
fig.update_yaxes(tickfont=dict(size=11)); fig.update_annotations(font_size=13); fig.update_layout(margin=dict(l=100))
put("cities", 6, "Which cities are above or below 2019?", fig, 460, f"Change in passengers, last 12 months vs 2019; only the {len(cty)} cities with at least 0.5 million passengers in 2019",
    how="Three short lists instead of one long one: every large city still below 2019, the six metros, and the ten non-metro cities that grew the most. Bars show the change in passengers (all routes touching the city) against 2019; hover for the passenger counts.",
    tells="Where the post-COVID market lost ground and where it found new passengers.",
    obs=[f"Only {len(belowC)} of {len(cty)} large cities are still below 2019" + (f", led by {cl(belowC.index[0])} ({belowC.pct.iloc[0]:+.0f}%)." if len(belowC) else "."),
         f"The fastest growth is in smaller cities: {', '.join(f'{cl(c)} ({v:+.0f}%)' for c, v in gainC.pct.sort_values(ascending=False).head(3).items())}, far ahead of the best metro, {cl(metC.index[-1])} ({metC.pct.iloc[-1]:+.0f}%)."])

# ---------------- SECTION 7: INTERNATIONAL FLIGHTS (Indian airlines), against the same pre-COVID baseline ----------------
IBM = {k: seasonal(PI, k) for k in ("rpk",)}
ipre_share = PI["rpk"].loc[PRE].sum() / PI["rpk"].loc[PRE].sum().sum()
IBASE = [g for g in PI["rpk"].columns if (PI["rpk"].loc[PRE, g] > 0).sum() == len(PRE) and ipre_share[g] >= MIN_BASE_SHARE and PI["rpk"].loc[W12, g].fillna(0).sum() > 0]
dom = {g: P["rpk"].loc[W12, g].sum() / BM["rpk"][g].sum() * 100 for g in IBASE if g in BASE}
intl = {g: PI["rpk"].loc[W12, g].fillna(0).sum() / IBM["rpk"][g].sum() * 100 for g in IBASE}
gI = sorted(dom, key=lambda g: -intl[g])
fig = go.Figure()
for g in gI:
    fig.add_trace(go.Scatter(x=[dom[g], intl[g]], y=[g, g], mode="lines", line=dict(color="#c8c8c8", width=5), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=[dom[g] for g in gI], y=gI, mode="markers+text", name="Domestic", marker=dict(symbol="circle", size=13, color="#333"), text=[f"{dom[g]:.0f}%" for g in gI], textposition="bottom center"))
fig.add_trace(go.Scatter(x=[intl[g] for g in gI], y=gI, mode="markers+text", name="International", marker=dict(symbol="diamond", size=13, color="#2166AC"), text=[f"{intl[g]:.0f}%" for g in gI], textposition="top center"))
fig.add_vline(x=100, line=dict(color="#222", dash="dash")); fig.update_yaxes(range=[len(gI) - 0.4, -0.7]); fig.update_xaxes(title="Passenger-km in the last 12 months, % of pre-COVID level"); fig.update_layout(margin=dict(l=130))
put("intl_dumbbell", 7, "Domestic vs international recovery", fig, 360, "Airlines with a full pre-COVID international record that still fly abroad",
    how="For each airline, the circle is its domestic passenger-km in the last 12 months as % of its pre-COVID level and the diamond the same for international flights; the dashed line is 100%.",
    tells="Whether airlines recovered more at home or abroad.",
    obs=[f"IndiGo's international flying is at {intl['IndiGo']:.0f}% of its pre-COVID level, far ahead of its domestic {dom['IndiGo']:.0f}%: it used the post-COVID years to build an international network.",
         f"SpiceJet is below its pre-COVID level on both (domestic {dom['SpiceJet']:.0f}%, international {intl['SpiceJet']:.0f}%): its shrinkage is company-wide, not a domestic-market problem."])
ir = pd.DataFrame({g: PI["rpk"][g] / seas(IBM["rpk"], g) * 100 for g in IBASE}).rolling(3).mean().loc["2017-06-01":]
fig = go.Figure([go.Scatter(x=ir.index, y=ir[g], name=g, mode="lines", line=dict(color=gc(g), width=2.5)) for g in IBASE])
fig.add_vrect(x0="2020-04-01", x1="2022-03-01", fillcolor="#eeeeee", line_width=0, layer="below", annotation_text="no scheduled international flights", annotation_position="top left")
fig.add_hline(y=100, line=dict(color="#222", dash="dash")); fig.update_yaxes(title="% of pre-COVID level"); fig.update_xaxes(title="Month")
fbI = {g: ir.loc["2022-04-01":, g][lambda s: s >= 100].index.min() for g in IBASE}
put("intl_lines", 7, "International passenger-km over time", fig, 420, "3-month averages",
    how="Each line is an airline's international passenger-km as a % of the same calendar month before COVID. The grey band is the ban on scheduled international flights (April 2020 to March 2022).",
    tells="When each airline's international flying came back, and how far it overshot.",
    obs=["; ".join((f"{g} back above its pre-COVID level in {d:%b %Y}" if pd.notna(d) else f"{g} never back") + f", now at {ir[g].dropna().iloc[-1]:.0f}%" for g, d in fbI.items()) + ". SpiceJet got back briefly and then fell away, the same story as its domestic flying.",
         "Scheduled international flights resumed only at the end of March 2022, almost two years after domestic, so it is the clearest test of who had aircraft and money to expand once the ban lifted."])


## 11. Relationships, prediction and conclusions

In [ ]:
# ---------------- SECTION 8: RELATIONSHIPS AND PREDICTION ----------------
KEYS = list(MET); LBL = {k: MET[k][0] for k in KEYS}
CM = pd.DataFrame(np.nan, index=KEYS, columns=KEYS); CN = CM.copy()
for a_ in KEYS:
    for b_ in KEYS:
        if a_ == b_: CN.loc[a_, b_] = OBS[a_].notna().sum(); CM.loc[a_, b_] = 1.0; continue
        d_ = OBS[[a_, b_]].dropna(); CN.loc[a_, b_] = len(d_)
        if len(d_) >= 20: CM.loc[a_, b_] = spearman(d_[a_], d_[b_])
def within(a_, b_):     # median correlation inside each airline (its own months only)
    d_ = OBS[[a_, b_, "g"]].dropna(); return float(np.nanmedian([spearman(x[a_], x[b_]) for _, x in d_.groupby("g") if len(x) > 5]))
zc = CM.values.astype(float)
fig = go.Figure(go.Heatmap(z=zc, x=[LBL[k] for k in KEYS], y=[LBL[k] for k in KEYS], zmin=-1, zmax=1, colorscale=DIV, text=[[("" if np.isnan(v) else f"{v:+.2f}") for v in row] for row in zc],
                           texttemplate="%{text}", xgap=2, ygap=2, customdata=CN.values, colorbar=dict(title=dict(text="Rank<br>correlation")),
                           hovertemplate="%{y} vs %{x}: %{z:.2f} (n = %{customdata} airline-months)<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(tickangle=-30); fig.update_layout(margin=dict(l=170, b=110))
put("corr", 8, "Which metrics move together?", fig, 520, "Blue = rise together, orange = one rises as the other falls; five interesting pairs are plotted in the Other plots section",
    how="Each cell is the rank correlation between two metrics over every airline-month since mid-2023 (each airline's 12-month window, stepped one month at a time). +1 means the two always rank airlines in the same order, -1 the opposite order, 0 no relation. Pairs involving demand or capacity cover only the 4 airlines with a pre-COVID record.",
    tells="Which metrics carry the same information (and so double-count in the index) and which add something new.",
    obs=[f"Demand and capacity recovery are almost the same number ({CM.loc['dem', 'cap']:+.2f}): load factors barely moved, so passengers followed seats. Recovery was supply-led: airlines that put aircraft back got the passengers.",
         f"Load factor and productivity ({CM.loc['lf', 'prod']:+.2f}) also overlap strongly, while growth is almost unrelated to load factor ({CM.loc['lf', 'gro']:+.2f}) and productivity ({CM.loc['prod', 'gro']:+.2f}): how full an airline flies says nothing about whether it is growing."])

PAIRS = [("dem", "vol", "Recovered airlines are the steady ones"),
         ("gro", "vol", "Across airlines vs inside one airline: the sign flips"),
         ("dem", "gro", "The winners keep winning, but not month by month"),
         ("lf", "prod", "Fuller planes = more passenger-km per flying hour"),
         ("lf", "gro", "Full planes do not mean growth")]
PNOTE = {}
for a_, b_, t_ in PAIRS:
    PNOTE[(a_, b_)] = (CM.loc[a_, b_], within(a_, b_), int(CN.loc[a_, b_]))
GL = ORDER + [g for g in OBS.g.unique() if g not in ORDER]
def pair_data(a_, b_):
    xs, ys, ts = [], [], []
    for g in GL:
        d_ = OBS[OBS.g == g][[a_, b_, "T"]].dropna(); xs.append(d_[a_].tolist()); ys.append(d_[b_].tolist()); ts.append(d_["T"].dt.strftime("%b %Y").tolist())
    return xs, ys, ts
def pair_title(a_, b_, t_):
    r, w, n = PNOTE[(a_, b_)]
    return f"<b>{t_}</b><br><span style='font-size:12px;color:#555'>{LBL[a_]} (across) vs {LBL[b_]} (up): {r:+.2f} across all airline-months, {w:+.2f} inside a typical airline (n = {n})</span>"
a0, b0, t0 = PAIRS[0]; xs, ys, ts = pair_data(a0, b0)
fig = go.Figure([go.Scatter(x=xs[i], y=ys[i], text=ts[i], mode="markers", name=g, marker=dict(color=gc(g), symbol=gs(g), size=8, opacity=.8, line=dict(color="white", width=.5)),
                            hovertemplate=g + "<br>%{text}<br>x %{x:.1f}, y %{y:.1f}<extra></extra>") for i, g in enumerate(GL)])
btn = []
for a_, b_, t_ in PAIRS:
    xs_, ys_, ts_ = pair_data(a_, b_)
    btn.append(dict(label=f"{LBL[a_]} vs {LBL[b_]}", method="update", args=[{"x": xs_, "y": ys_, "text": ts_},
               {"xaxis.title.text": f"{LBL[a_]} ({SHORTU[a_]})", "yaxis.title.text": f"{LBL[b_]} ({SHORTU[b_]})", "annotations[0].text": pair_title(a_, b_, t_)}]))
fig.update_layout(updatemenus=[dict(type="dropdown", buttons=btn, x=0, y=1.0, xanchor="left", yanchor="bottom", bgcolor="white", font=dict(size=12))],
                  annotations=[dict(text=pair_title(a0, b0, t0), x=0.32, y=1.0, xref="paper", yref="paper", xanchor="left", yanchor="bottom", align="left", showarrow=False, font=dict(size=13))],
                  margin=dict(t=120), legend=dict(orientation="h", y=-0.2, x=0))
fig.update_xaxes(title=f"{LBL[a0]} ({SHORTU[a0]})"); fig.update_yaxes(title=f"{LBL[b0]} ({SHORTU[b0]})")
r_dv, w_dv, _ = PNOTE[("dem", "vol")]; r_gv, w_gv, _ = PNOTE[("gro", "vol")]; r_dg, w_dg, _ = PNOTE[("dem", "gro")]; r_lp, w_lp, _ = PNOTE[("lf", "prod")]; r_lg, w_lg, _ = PNOTE[("lf", "gro")]
put("pairs", 11, "Five pairs worth talking about", fig, 560, "Pick a pair from the drop-down; each dot is one airline in one month, coloured by airline",
    how="A scatter of two raw metrics. Each dot is an airline's 12-month window ending in one month since mid-2023, so each airline leaves a short trail. The heading gives two numbers: the correlation across all dots, and the typical correlation inside one airline's own trail. When the two differ, the pattern is about which airline you are, not about what happens over time.",
    tells="Whether a link in the heatmap is a real mechanism or just the difference between airlines (Simpson's paradox).",
    obs=[f"Demand recovery vs volatility ({r_dv:+.2f}): airlines above their pre-COVID level have steady months; those far below (SpiceJet, Alliance Air) swing wildly. Shrinking airlines are also unstable ones.",
         f"Growth vs volatility flips sign: {r_gv:+.2f} across airlines but {w_gv:+.2f} inside an airline. Between airlines the growers are the steady ones, but when any one airline grows fast its months get bumpier: rapid expansion is disruptive.",
         f"Demand recovery vs growth: {r_dg:+.2f} across airlines but {w_dg:+.2f} inside one. Airlines far above their pre-COVID level are also the fastest growers, but that is a fixed difference between airlines, not something that moves month to month.",
         f"Load factor vs productivity ({r_lp:+.2f} across, {w_lp:+.2f} inside): passenger-km per block hour is largely load factor in disguise, so the Operations pillar partly counts the same thing twice.",
         f"Load factor vs growth ({r_lg:+.2f}): Akasa Air grew fast with full planes while SpiceJet shrank (2023-25) with planes just as full. A high load factor alone is not a sign of health."])

XS = S0.loc[ORDER, COMMON].fillna(50)
ZL = linkage(XS.values, "ward"); leaf = [ORDER[i] for i in dendrogram(ZL, no_plot=True)["leaves"]]
DM = pd.DataFrame(squareform(pdist(XS.values)), index=ORDER, columns=ORDER).loc[leaf, leaf]
fig = go.Figure(go.Heatmap(z=DM.values, x=leaf, y=leaf, colorscale=[[0, "#08306b"], [0.5, "#6baed6"], [1, "#f7fbff"]], text=DM.round(0).values, texttemplate="%{text:.0f}",
                           xgap=2, ygap=2, colorbar=dict(title=dict(text="Distance")), hovertemplate="%{y} to %{x}: %{z:.0f}<extra></extra>"))
fig.update_yaxes(autorange="reversed"); fig.update_layout(margin=dict(l=130))
off = DM.where(~np.eye(len(DM), dtype=bool)).stack(); cl_ = off.idxmin(); far_ = off.idxmax()
put("distance", 11, "How far apart are the airlines?", fig, 460, "Dark = similar",
    how=f"Distance between two airlines = square root of the sum of squared score differences over the five metrics every airline has ({', '.join(LBL[k].lower() for k in COMMON)}). Two airlines 25 points apart on all five are about 56 apart.",
    tells="Which airlines behave alike regardless of size.",
    obs=[f"{cl_[0]} and {cl_[1]} are the most alike (distance {off[cl_]:.0f}); {far_[0]} and {far_[1]} the least ({off[far_]:.0f}).",
         "The dark blocks along the diagonal are peer groups; the next chart draws them as a tree."])

THR = (ZL[-3, 2] + ZL[-2, 2]) / 2
fig = ff.create_dendrogram(XS.values, labels=ORDER, orientation="left", linkagefun=lambda x: linkage(x, "ward"), color_threshold=THR,
                           colorscale=["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#A0522D", "#333333"])
fig.add_vline(x=THR, line=dict(dash="dot", color="#888")); fig.update_xaxes(title="Merge height (how much spread a merge adds)")
cl3 = fcluster(ZL, t=3, criterion="maxclust"); groups_ = [[g for g, k in zip(ORDER, cl3) if k == c] for c in sorted(set(cl3))]
put("dendro", 11, "Peer groups", fig, 420, "The dotted line cuts the tree into three groups",
    how="Built from the same distances: the two closest airlines are joined first, then the next closest airline or group, and so on. The further right a join happens, the less alike the two sides are.",
    tells="Natural peer groups to compare airlines within.",
    obs=["Peer groups: " + "; ".join(", ".join(v) for v in groups_) + ".",
         "The groups follow business model (jets vs turboprops) more than recovery: that is why the scorecard should be read within peer groups as well as overall."])

lab = {**LBL, **{p: p + " pillar" for p in PILLARS}, "Index": "Performance index"}
ps = PRED.sort_values(); xs, ys = [], []
for k, v in ps.items(): xs += [0, v, None]; ys += [lab[k], lab[k], None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#bbb", width=3), hoverinfo="skip", showlegend=False),
                 go.Scatter(x=ps.values, y=[lab[k] for k in ps.index], mode="markers+text", text=[f"{v:+.2f}" for v in ps.values], textposition=["middle right" if v >= 0 else "middle left" for v in ps.values],
                            marker=dict(size=13, color=["#2166AC" if v > .3 else ("#B35806" if v < -.1 else "#999") for v in ps.values], symbol=["circle" if v > .3 else ("x" if v < -.1 else "circle-open") for v in ps.values],
                                        line=dict(color="#222", width=1)), customdata=[int(NOBS[k]) for k in ps.index], hovertemplate="%{y}: %{x:+.2f} (n = %{customdata})<extra></extra>", showlegend=False)])
fig.add_vline(x=0, line=dict(color="#222", width=1)); fig.update_xaxes(title="Rank correlation with the next 6 months of passenger-km growth", range=[-.5, 1.05])
bestm = ps.drop(["Index"] + PILLARS).idxmax()
put("lollipop", 8, "Which metrics predict the next six months?", fig, 480, "Filled blue = useful (above 0.3); open grey = no signal",
    how="For every airline-month we take the metric as it stood then and the airline's passenger-km growth over the following six months, and rank-correlate the two. +1 = the metric would have ranked airlines exactly in the order they then grew.",
    tells="Which parts of the scorecard look forward and which only describe the present.",
    obs=[f"{LBL[bestm]} is the best guide ({ps[bestm]:+.2f}): growth continues. The Momentum pillar ({PRED['Momentum']:+.2f}) and Recovery pillar ({PRED['Recovery']:+.2f}) carry real signal.",
         f"The Operations pillar predicts nothing ({PRED['Operations']:+.2f}): full, efficient planes do not tell you whether an airline will grow. The load factor vs growth pair in the Other plots section shows the same thing."])

labs = PILLARS + ["Index"]; peers = BT_P.drop(index="Go First"); xs, ys = [], []
for p in labs:
    v = peers[p].dropna(); xs += [v.min(), v.max(), None]; ys += [p, p, None]
fig = go.Figure([go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="#ccc", width=12), name=f"Range of the other {len(peers)} airlines", hoverinfo="skip"),
                 go.Scatter(x=[peers[p].median() for p in labs], y=labs, mode="markers", marker=dict(symbol="line-ns", size=18, color="#555", line=dict(width=3, color="#555")), name="Their median"),
                 go.Scatter(x=[BT_P.loc["Go First", p] for p in labs], y=labs, mode="markers+text", text=[f"{BT_P.loc['Go First', p]:.0f}" for p in labs], textposition="top center",
                            marker=dict(size=15, color=gc("Go First"), symbol="diamond", line=dict(color="#222", width=1)), name="Go First")])
fig.update_xaxes(title=f"Score as of {BT_T:%B %Y} (Go First's last full month), using only data up to then", range=[0, 100]); fig.update_yaxes(autorange="reversed")
put("backtest", 8, "Would the metrics have flagged Go First?", fig, 420, f"Peers scored that month: {', '.join(peers.index)}",
    how=f"We re-run the whole scorecard as if it were {BT_T:%B %Y}, using only data available then, and place Go First (diamond) against the range of the other airlines (grey bar, with their median).",
    tells="Whether the scorecard is an early-warning system for airline failure.",
    obs=[f"No. In its last full month Go First ranked {BT_rank['Go First']} of {len(BT_P)} on the index ({BT_P.loc['Go First', 'Index']:.0f}), with the second-best operations ({BT_P.loc['Go First', 'Operations']:.0f}). Its recovery ({BT_P.loc['Go First', 'Recovery']:.0f}) and momentum ({BT_P.loc['Go First', 'Momentum']:.0f}) were weak, but SpiceJet scored lower on both ({BT_P.loc['SpiceJet', 'Recovery']:.0f} and {BT_P.loc['SpiceJet', 'Momentum']:.0f}) and is still flying. Its collapse in May 2023 (engine problems and a cash crunch, not falling demand) did not show up in its traffic beforehand.",
         "May 2023 itself is left out on purpose: Go First flew only the first days of that month, which would have made it look worst for the wrong reason. The low end of the Operations bar is Alliance Air (turboprops), a real value, not missing data."])

# ---------------- SECTION 9: CONCLUSIONS (with the scorecard gauges) ----------------
PILp = YEARLY[YEARS[-2]] if len(YEARLY[YEARS[-2]]) else None
fig = make_subplots(rows=2, cols=4, specs=[[{"type": "indicator"}] * 4] * 2, vertical_spacing=0.38)
for k, g in enumerate(ORDER):
    d = dict(reference=round(float(PILp.loc[g, "Index"])), valueformat=".0f") if (PILp is not None and g in PILp.index) else {}
    fig.add_trace(go.Indicator(mode="gauge+number" + ("+delta" if d else ""), value=round(float(PIL0.loc[g, "Index"])), number=dict(valueformat=".0f"), delta=d, title=dict(text=g, font=dict(size=14)),
        gauge=dict(axis=dict(range=[0, 100]), bar=dict(color=gc(g), thickness=.55), steps=[dict(range=[0, 40], color="#efefef"), dict(range=[40, 60], color="#e2e2e2"), dict(range=[60, 100], color="#d4d4d4")])),
        row=k // 4 + 1, col=k % 4 + 1)
fig.update_layout(margin=dict(t=90, b=20))
dlt = {g: PIL0.loc[g, "Index"] - PILp.loc[g, "Index"] for g in ORDER if PILp is not None and g in PILp.index}
put("gauges", 10, "Final scores", fig, 500, f"Arrows show the change since December {YEARS[-2].year}",
    how=f"One dial per airline showing its performance index for the 12 months to {LAST:%B %Y}; the shaded bands mark weak (below 40), typical (40-60) and strong (above 60).",
    tells="The headline result of the scorecard in one view.",
    obs=[f"Only IndiGo is in the strong band ({PIL0.loc['IndiGo', 'Index']:.0f}); Alliance Air is far behind everyone ({PIL0.loc['Alliance Air', 'Index']:.0f}).",
         f"Biggest move since December {YEARS[-2].year}: {max(dlt, key=lambda g: abs(dlt[g]))} ({dlt[max(dlt, key=lambda g: abs(dlt[g]))]:+.0f} points)."])

tot19, tot12m = mk.loc[PRE].mean() / 1e6, mk.loc[W12].mean() / 1e6
REC = {g: ROLL.loc["2020-07-01":, g].dropna()[lambda s: s >= 1].index.min() for g in BASE}
firstg = sorted([(g, d) for g, d in REC.items() if pd.notna(d)], key=lambda t: t[1]); never = [g for g, d in REC.items() if pd.isna(d)]
# load factor recovery: 3-month load factor at or above the same months before COVID
LFB = {g: lf_of(BM["rpk"][g], BM["ask"][g]) for g in BASE}
LFR = {}
for g in BASE:
    r3 = lf_of(P["rpk"][g].rolling(3).sum(), P["ask"][g].rolling(3).sum()); b3 = lf_of(seas(BM["rpk"], g).rolling(3).sum(), seas(BM["ask"], g).rolling(3).sum())
    hit = (r3 - b3).loc["2020-07-01":].dropna(); hit = hit[hit >= 0]; LFR[g] = hit.index.min() if len(hit) else pd.NaT
lf_first = sorted([(g, d) for g, d in LFR.items() if pd.notna(d)], key=lambda t: t[1]); lf_never = [g for g, d in LFR.items() if pd.isna(d)]
sr0 = P["rpk"].loc[PRE].sum() / P["rpk"].loc[PRE].sum().sum(); sr12 = P["rpk"].loc[W12].sum() / P["rpk"].loc[W12].sum().sum()
LF_PRE = {g: lf_of(P["rpk"].loc[PRE, g].sum(), P["ask"].loc[PRE, g].sum()) for g in BASE}
lf_chg = {g: M0.loc[g, "lf"] - LF_PRE[g] for g in BASE if g in M0.index}
# Jet Airways, from the data: its share just before it stopped, and IndiGo's share either side
j_q1 = P["pax"].loc["2019-01-01":"2019-03-01"]; j_sh = j_q1["Jet Airways"].sum() / j_q1.sum().sum() * 100
i_18 = P["pax"].loc["2018-01-01":"2018-12-01"]; i_h2 = P["pax"].loc["2019-07-01":"2019-12-01"]
i_sh18, i_sh19 = i_18["IndiGo"].sum() / i_18.sum().sum() * 100, i_h2["IndiGo"].sum() / i_h2.sum().sum() * 100
j_last = P["pax"]["Jet Airways"][P["pax"]["Jet Airways"] > 0].index.max()
ANSWER = [f"Passenger numbers, first to last: " + ", ".join(f"{g} ({d:%b %Y})" for g, d in firstg) + f". Never: {', '.join(never)}.",
          "Load factor (first month the 3-month load factor matched the same months before COVID), first to last: " + ", ".join(f"{g} ({d:%b %Y})" for g, d in lf_first) + (f". Never: {', '.join(lf_never)}." if lf_never else "."),
          f"Still above the pre-COVID passenger level today: {' and '.join(g for g in BASE if STATUS[g] == 'Recovered and holding')}. Go First recovered and then stopped flying (May 2023); SpiceJet never got back."]
CONCL = [
  ("The market is well above its pre-COVID level; most airlines that flew before COVID are not.",
   f"Passengers per month are {(tot12m / tot19 - 1) * 100:+.0f}% on the April 2017 to March 2020 average, but only 2 of the 5 airlines with a full pre-COVID record are above their own level today."),
  ("Load factors recovered only partly.", "Change in average load factor, last 12 months vs April 2017 to March 2020: " + "; ".join(f"{g} {v:+.1f} points" for g, v in sorted(lf_chg.items(), key=lambda t: -t[1])) + "."),
  ("One airline took the gains.", f"IndiGo's share of passenger-km rose from {sr0['IndiGo'] * 100:.0f}% before COVID to {sr12['IndiGo'] * 100:.0f}%; IndiGo and Air India Group now fly {(sr12['IndiGo'] + sr12['Air India Group']) * 100:.0f}% of all passenger-km."),
  ("Jet Airways left before COVID, and IndiGo was already filling the gap.",
   f"Jet Airways carried {j_sh:.0f}% of domestic passengers in January-March 2019 and its last month in the data is {j_last:%B %Y} (it suspended all flights on 17 April 2019; India's Supreme Court ordered its liquidation on 7 November 2024). IndiGo's passenger share went from {i_sh18:.0f}% in 2018 to {i_sh19:.0f}% in July-December 2019, before COVID began. Because Jet stopped inside our pre-COVID baseline window, it has no full baseline and is not scored."),
  ("Recovery was supply-led.", f"Demand and capacity recovery are almost perfectly correlated ({CM.loc['dem', 'cap']:+.2f}): the airlines that put aircraft back got the passengers; SpiceJet and Alliance Air shrank their own fleets while keeping their planes full."),
  ("Growth came from smaller cities.", f"Routes between non-metro cities are at {pct_k['Other city to other city']:.0f}% of 2019; metro-to-metro routes at {pct_k['Metro to metro']:.0f}%. {len(new)} new routes opened since 2019."),
  ("Scores: IndiGo first, Alliance Air last.", f"IndiGo {PIL0.loc['IndiGo', 'Index']:.0f}, Akasa Air {PIL0.loc['Akasa Air', 'Index']:.0f}, Air India Group {PIL0.loc['Air India Group', 'Index']:.0f}, Fly91 {PIL0.loc['Fly91', 'Index']:.0f}, Star Air {PIL0.loc['Star Air', 'Index']:.0f}, SpiceJet {PIL0.loc['SpiceJet', 'Index']:.0f}, Alliance Air {PIL0.loc['Alliance Air', 'Index']:.0f}."),
  ("Growth predicts growth; traffic data cannot predict a collapse.", f"Traffic growth is the best predictor of the next six months ({PRED['gro']:+.2f}), but Go First ranked {BT_rank['Go First']} of {len(BT_P)} in its last full month before it stopped flying."),
]
holding = [g for g in BASE if STATUS[g] == "Recovered and holding"]
lf_best = max(lf_chg, key=lf_chg.get)
# ---------------- CONCLUSIONS: seven numbered findings, every number computed above ----------------
from itertools import groupby
NUMW = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 7: "seven", 8: "eight", 9: "nine"}
pkm_g = (P["rpk"].loc[W12].sum(axis=1).mean() / P["rpk"].loc[PRE].sum(axis=1).mean() - 1) * 100      # passenger-km, average month vs pre-COVID
reg_txt = "; ".join(" and ".join(g for g, _ in grp) + f" in {d:%B %Y}" for d, grp in ((d, list(gr)) for d, gr in groupby(firstg, key=lambda t: t[1])))
hhi0, hhi12 = float(((sr0 * 100) ** 2).sum()), float(((sr12 * 100) ** 2).sum())                    # airline concentration (HHI), pre-COVID and last 12 months
lf_show = [lf_best] + [g for g in ("IndiGo", "SpiceJet") if g != lf_best]
pkm_bt = BT_P.loc["Go First", "Index"]
CONCL_ITEMS = [
  (f"Passenger numbers: the market is back, but only {NUMW.get(len(holding), len(holding))} of {NUMW.get(len(BASE), len(BASE))} pre-COVID airlines are above their old level.",
   f"Passengers per month over the last 12 months are {(tot12m / tot19 - 1) * 100:.0f}% above the pre-COVID average (April 2017 to March 2020), and passenger-km {pkm_g:.0f}% above. "
   f"Of the {len(BASE)} airlines that flew every pre-COVID month, {len(firstg)} regained their pre-COVID level ({reg_txt}), {len(holding)} are still above it, and {' and '.join(never)} never got back."),
  ("Load factors: almost nobody is back to the pre-COVID level.",
   f"Only {lf_best} fills a larger share of seats than before COVID (" + "; ".join(f"{g} {lf_chg[g]:+.1f} points" for g in lf_show) + "). "
   "Capacity came back faster than demand, so planes are a little emptier even where passengers grew."),
  ("The market concentrated on airlines but not on routes.",
   f"IndiGo's share of passenger-km went from {sr0['IndiGo'] * 100:.0f}% to {sr12['IndiGo'] * 100:.0f}% and the airline concentration index (HHI) from about {round(hhi0, -1):,.0f} to about {round(hhi12, -1):,.0f} (above 2,500 is highly concentrated). "
   f"On routes, the busiest 10% carry {top10(RT.p19.values):.0f}% of passengers in 2019 and {top10(RT.now.values):.0f}% now: growth came from many routes, including {len(new)} new ones."),
  ("Recovery was led by smaller cities, not the metros.",
   f"Metro-to-metro routes are at {pct_k['Metro to metro']:.0f}% of 2019 while routes between smaller cities are at {pct_k['Other city to other city']:.0f}%. "
   f"{len(belowC)} of {len(cty)} large cities are still below 2019, including {cl(belowC.index[0])} ({belowC.pct.iloc[0]:+.0f}%) and {cl(metC.index[0])} ({metC.pct.iloc[0]:+.0f}%)."),
  ("International flying is well above its pre-COVID level, driven by IndiGo.",
   f"Over the last 12 months IndiGo's international passenger-km is at {intl['IndiGo']:.0f}% of its pre-COVID level against {dom['IndiGo']:.0f}% at home; "
   f"Air India Group's is at {intl['Air India Group']:.0f}% against {dom['Air India Group']:.0f}% at home, and SpiceJet's at {intl['SpiceJet']:.0f}%."),
  (f"The index, with countermeasures for new airlines, puts {top} first.",
   f"IndiGo scores {PIL0.loc['IndiGo', 'Index']:.0f}, Akasa Air {PIL0.loc['Akasa Air', 'Index']:.0f}, Air India Group {PIL0.loc['Air India Group', 'Index']:.0f} and Alliance Air {PIL0.loc['Alliance Air', 'Index']:.0f}. "
   f"Without the countermeasures Akasa Air would lead ({cmp.loc['Akasa Air', 'old']:.0f}) because it was scored only on its two strongest pillars."),
  ("Growth predicts growth; efficiency does not.",
   f"Traffic growth lines up best with the next six months ({PRED['gro']:+.2f}), then the Recovery pillar ({PRED['Recovery']:+.2f}); the Operations pillar scores {PRED['Operations']:+.2f}. "
   f"In a backtest Go First ranked {BT_rank['Go First']} of {len(BT_P)} in its last full month (index {pkm_bt:.0f}), and SpiceJet scored lower on recovery and momentum yet is still flying, so traffic data alone did not flag the collapse."),
]
LIMITS = ["Scores are relative: 50 means typical of the airlines scored that month, not an absolute standard. Only seven airlines are scored.",
          "Operations metrics favour big jets on long trips; turboprop operators (Star Air, Fly91, Alliance Air) score low there regardless of management.",
          "The countermeasure settings (floors, 36-month maturity, 50 for a missing pillar) are judgment calls; the old-vs-adjusted chart shows their effect.",
          "City-pair data has no airline column, and some city merges are judgment calls (two Goa airports, Navi Mumbai into Mumbai, Hindon into Ghaziabad).",
          "Months with no scheduled international flights (April 2020 to early 2022) are treated as zero; air-bubble flights in that period are not in this series.",
          f"DGCA data is provisional. Airline analysis stops at {LAST:%B %Y} (last month reconciling to DGCA's total within 2%); city-pair data ends {LASTC:%B %Y}."]
CONCL_HTML = '<div class="blk"><div class="doc"><ol class="concl">\n' + "".join(f"<li><b>{_html.escape(h, quote=False)}</b> {_html.escape(b, quote=False)}</li>\n" for h, b in CONCL_ITEMS) + "</ol></div></div>"
for h, b in CONCL_ITEMS: print(h, "|", b)


## 12. Build the HTML page (charts are Plotly figures; header, jump buttons, conclusions and credits are HTML)


In [12]:
# ASSEMBLE ONE SELF-CONTAINED HTML PAGE. Charts, notes and section headings are Plotly figures; the page header, the jump buttons,
# the conclusions and the credits are plain HTML. Plotly.js is embedded, so the file opens offline.
CFG = dict(displaylogo=False, responsive=True, modeBarButtonsToRemove=["lasso2d", "select2d"])
STATIC = dict(displayModeBar=False, responsive=True)     # text panels: no toolbar

# Move three supporting charts out of "Other plots" next to the charts they belong with: key -> (section, place after this chart)
PLACE = {"sunburst": (5, "sizeperf"), "distance": (9, "bump"), "dendro": (9, "distance")}
for _k, (_s, _after) in PLACE.items():
    _v = FIGS.pop(_k); _v["sec"] = _s; _it = list(FIGS.items()); _i = [k for k, _ in _it].index(_after) + 1
    _it.insert(_i, (_k, _v)); FIGS.clear(); FIGS.update(_it)

# HIGHLIGHTS: four KPI tiles, each inside its own outlined box
HEADER = go.Figure()
kp = [((tot12m / tot19 - 1) * 100, "%", "Passengers per month<br>vs pre-COVID average"),
      (sr12["IndiGo"] * 100, "%", "IndiGo's share of<br>passenger-km (was %.0f%%)" % (sr0["IndiGo"] * 100)),
      (sum(STATUS[g] == "Recovered and holding" for g in BASE), f" of {len(BASE)}", "pre-COVID airlines above<br>their own pre-COVID level"),
      (PIL0.loc[top, "Index"], "", f"Top index: {top}<br>(0-100, 50 = typical)")]
BOX = [(0.005, 0.245), (0.255, 0.495), (0.505, 0.745), (0.755, 0.995)]
for i, ((v, suf, lab_), (x0, x1)) in enumerate(zip(kp, BOX)):
    HEADER.add_trace(go.Indicator(mode="number", value=float(v), number=dict(suffix=suf, valueformat="+.0f" if i == 0 else ".0f", font=dict(size=40, color="#2166AC")),
                                  title=dict(text=lab_, font=dict(size=13, color="#444")), domain=dict(x=[x0 + 0.02, x1 - 0.02], y=[0.0, 0.9])))
    HEADER.add_shape(type="rect", xref="paper", yref="paper", x0=x0, x1=x1, y0=0, y1=1, line=dict(color="#2166AC", width=1.5), fillcolor="white", layer="below")
HEADER.update_layout(template="plotly_white", font=FONT, height=135, margin=dict(l=4, r=4, t=4, b=4), paper_bgcolor="rgba(0,0,0,0)")

def sec_fig(n, t):
    f = go.Figure(); f.add_annotation(x=0, y=0.5, xref="paper", yref="paper", xanchor="left", yanchor="middle", showarrow=False, text=f"<b>{n}. {t}</b>", font=dict(size=22, color="#111"))
    f.update_layout(template="plotly_white", height=64, margin=dict(l=10, r=10, t=10, b=6), xaxis=dict(visible=False), yaxis=dict(visible=False), paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
                    shapes=[dict(type="line", xref="paper", yref="paper", x0=0, x1=1, y0=0, y1=0, line=dict(color="#2166AC", width=2))])
    return f
INTRO = {2: [MET_TABLE, METHOD_TEXT],
         3: [text_fig([("Pre-COVID level", f"April 2017 to March 2020: three full years ending just before the lockdown. A month is compared with the same calendar month averaged over those three years, so seasonality cancels out. Only {len(BASE)} airlines ({', '.join(BASE)}) flew every one of those 36 months with at least 0.5% of passengers; the first chart below explains why the others drop out.")])],
         6: [text_fig([("About this section", "The city-pair file has no airline column, so this section is about routes and cities rather than airlines. Routes are compared with 2019 (the last full year before COVID) because the 2017 city-pair data is incomplete.")])],
         11: [text_fig([("About this section", "Extra charts that support the main story but are not needed to follow it.")])]}

# CREDITS AND LIMITATIONS (plain HTML)
CREDITS_HTML = f"""<div class="blk" id="credits"><div class="doc">
<h2>Credits and limitations</h2>
<h3>Data and sources</h3>
<ul>
<li>DGCA (Government of India), monthly domestic traffic by airline (Excel files), parsed by the team.</li>
<li>DGCA city-pair traffic as compiled in the Vonter/india-aviation-traffic GitHub repository (pinned commit e197170), file {CITY_FILE.name}. The Vonter GitHub source operates under an Open Database License (ODbL), according to the repository README.</li>
<li>Charts: Plotly. Generated {TODAY}.</li>
</ul>
<h3>Contribution split</h3>
<ul>
<li>Subranshu DA25B050 - Scraped data from DGCA website using selenium, Came up with certain visualisation</li>
<li>Pranauv G DA25B035 - Introduced city-pair metrics and related visualisations. Also international flight data to improve data visualisation</li>
<li>Sashwath J S DA25B043 - Scraped data from Github source, helped to come with metrics derived from data columns</li>
</ul>
<p>Moreover, all 3 collaborated on data cleaning, improving graphs and other refinements.</p>
<h3>LLM Usage</h3>
<p>Claude (Anthropic) - Was used to verify and debug scraping and cleaning code. Helped with building of website, identifying errors and formatting issues at various points.</p>
<h3>robots.txt</h3>
<p>There was no robots.txt in either source. The DGCA website was free to collect data from, as it is a Government of India website.</p>
<h3>Limitations</h3>
<ul>
""" + "".join(f"<li>{_html.escape(t, quote=False)}</li>\n" for t in LIMITS) + """</ul>
</div></div>"""

# JUMP BUTTONS: one square button per part of the page, in page order (Credits and limitations sits between 10 and 11)
PARTS = [("s1", "1", "Data"), ("s2", "2", "Metrics"), ("s3", "3", "Recovery"), ("s4", "4", "Operations"), ("s5", "5", "Momentum & share"), ("s6", "6", "Routes"),
         ("s7", "7", "International"), ("s8", "8", "Relation-ships"), ("s9", "9", "Scorecard"), ("s10", "10", "Conclu-sions"), ("credits", "C", "Credits & limitations"), ("s11", "11", "Other plots")]
PART_NAME = {"s1": "The data", "s2": "The seven metrics", "s3": "Recovery against the pre-COVID level", "s4": "Operations", "s5": "Momentum and market share", "s6": "The route network",
             "s7": "International flights", "s8": "Relationships and prediction", "s9": "The scorecard", "s10": "Conclusions", "credits": "Credits and limitations", "s11": "Other plots"}
NAV = '<nav class="tabs">' + "".join(f'<button class="tab" data-p="{k}" title="{n}. {PART_NAME[k]}" aria-label="{PART_NAME[k]}"><span class="n">{n}</span><span class="l">{l.replace("-", "&#173;")}</span></button>' for k, n, l in PARTS) + "</nav>"

def block(anchor, f):
    div = f.to_html(full_html=False, include_plotlyjs=False, config=STATIC if (f.layout.meta == "text" or anchor and anchor.startswith("s") and anchor[1:].isdigit()) else CFG, auto_play=False)
    return f'<div class="blk"{f" id={chr(34)}{anchor}{chr(34)}" if anchor else ""}>{div}</div>'
def section_html(n):
    out = [block(f"s{n}", sec_fig(n, SEC[n]))] + [block(None, f) for f in INTRO.get(n, [])]
    for k, v in FIGS.items():
        if v["sec"] == n: out += [block(f"c-{k}", v["fig"]), block(None, notes_fig(v))]
    if n == 10: out.append(CONCL_HTML)
    return f'<section class="panel" id="p-s{n}">' + "".join(out) + "</section>"
SECTIONS = [section_html(n) for n in range(1, 11)] + ['<section class="panel" id="p-credits">' + CREDITS_HTML + "</section>", section_html(11)]

PAGE_TITLE = "Airline recovery analysis post COVID and analysis of other performance factors for several Indian domestic airlines"
TOP = (f'<header class="top"><h1>{PAGE_TITLE}</h1>\n'
       '<p>The initial scope was only airline recovery post COVID, but it then expanded to a broader scope.</p>\n'
       f'<p>Which airlines regained their pre-COVID passenger numbers and load factors first, and which never did? Scheduled domestic flights, January 2017 to {LAST:%B %Y}. Source: DGCA monthly airline statistics and DGCA city-pair statistics.</p></header>\n'
       '<div class="hl-label">Highlights</div><div id="hl">' + block(None, HEADER) + "</div>")

CSS = """<style>
*{box-sizing:border-box}
body{margin:0;background:#eef1f5;font-family:Arial,sans-serif;color:#222}
main{max-width:1180px;margin:0 auto;padding:16px}
.blk{background:#fff;border-radius:8px;margin:0 0 10px;overflow:hidden;scroll-margin-top:10px}
header.top{background:#fff;border-radius:8px;margin:0 0 10px;padding:18px 24px;border-left:4px solid #2166AC}
header.top h1{margin:0 0 8px;font-size:28px;line-height:1.25;color:#111}
header.top p{margin:4px 0 0;font-size:14px;color:#555;line-height:1.45}
.hl-label{font-size:16px;font-weight:bold;color:#111;margin:14px 2px 6px}
#hl{margin:0 0 10px}#hl .blk{background:transparent;margin:0;border-radius:0}
nav.tabs{position:sticky;top:0;z-index:50;background:#eef1f5;padding:6px 0 8px;margin:0 0 6px;display:flex;gap:6px;overflow-x:auto}
.tab{flex:1 0 84px;max-width:96px;aspect-ratio:1/1;min-width:84px;border:1.5px solid #2166AC;background:#fff;color:#2166AC;border-radius:0;font-family:Arial,sans-serif;cursor:pointer;display:flex;flex-direction:column;align-items:center;justify-content:center;gap:4px;padding:4px}
.tab .n{font-size:22px;font-weight:bold;line-height:1}
.tab .l{font-size:11px;line-height:1.15;text-align:center}
.tab:hover{background:#e6eef7}
.tab.on{background:#2166AC;color:#fff}
.panel{scroll-margin-top:112px}
.doc{padding:18px 24px 14px;font-size:14px;line-height:1.55;color:#222}
.doc h2{margin:0 0 12px;padding-bottom:10px;font-size:22px;color:#111;border-bottom:2px solid #2166AC}
.doc h3{margin:18px 0 6px;font-size:16px;color:#111}
.doc p{margin:6px 0}.doc ol.concl li{margin:10px 0}.doc ul,.doc ol{margin:6px 0;padding-left:26px}.doc li{margin:3px 0}
</style>"""
FONT_PATCH = """<script>(function(){var np=Plotly.newPlot;Plotly.newPlot=function(id,d,l,c){l=l||{};l.font=Object.assign({},l.font,{family:'Arial'});l.hoverlabel=Object.assign({},l.hoverlabel,{font:Object.assign({},(l.hoverlabel||{}).font,{family:'Arial'})});return np.call(this,id,d,l,c);};})();</script>"""      # every chart uses Arial, including ones whose layout sets no font
NAV_JS = """<script>
(function(){
var tabs=[].slice.call(document.querySelectorAll('.tab')),panels=[].slice.call(document.querySelectorAll('.panel'));
function mark(k){tabs.forEach(function(t){t.classList.toggle('on',t.dataset.p===k);});}
tabs.forEach(function(t){t.addEventListener('click',function(){var p=document.getElementById('p-'+t.dataset.p);p.scrollIntoView({behavior:'smooth',block:'start'});mark(t.dataset.p);try{history.replaceState(null,'','#'+t.dataset.p);}catch(e){}});});
function onScroll(){var nav=document.querySelector('nav.tabs'),y=nav.getBoundingClientRect().bottom+20,cur=null;
  panels.forEach(function(p){if(p.getBoundingClientRect().top<=y)cur=p.id.slice(2);});
  if(window.innerHeight+window.pageYOffset>=document.body.scrollHeight-4)cur=panels[panels.length-1].id.slice(2);
  mark(cur);}
var tk=false;window.addEventListener('scroll',function(){if(!tk){tk=true;requestAnimationFrame(function(){onScroll();tk=false;});}});
var h=(location.hash||'').slice(1),el=h&&(document.getElementById('p-'+h)||document.getElementById(h));if(el)el.scrollIntoView();
onScroll();
})();
</script>"""          # jump buttons scroll to their section; the button of the section on screen is highlighted

page = ('<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n'
        "<title>Airline recovery analysis post COVID</title>\n" + CSS + "\n<script>" + get_plotlyjs() + "</script>" + FONT_PATCH + "</head><body><main>"
        + TOP + NAV + "".join(SECTIONS) + "</main>" + NAV_JS + "</body></html>")
OUT_FILE = OUT / f"dgca_dashboard_v5_{TODAY}.html"
OUT_FILE.write_text(page, encoding="utf-8")
print(f"saved {OUT_FILE.name}: {OUT_FILE.stat().st_size / 1e6:.1f} MB | {len(FIGS)} charts")


saved dgca_dashboard_v5_2026-11-05.html: 6.0 MB | 38 charts


## 13. The charts in the notebook, each with its notes panel

In [ ]:
# SHOW EVERY CHART IN THE NOTEBOOK TOO, in page order, each followed by its notes panel (set SHOW_FIGURES = False in the first cell to skip)
from IPython.display import HTML
if SHOW_FIGURES:
    display(Markdown(f"# {PAGE_TITLE}")); HEADER.show(config=CFG)
    for n in list(range(1, 11)) + ["credits", 11]:
        if n == "credits":
            display(HTML(CREDITS_HTML)); continue
        display(Markdown(f"## {n}. {SEC[n]}"))
        for f in INTRO.get(n, []): f.show(config=CFG)
        for k, v in FIGS.items():
            if v["sec"] != n: continue
            v["fig"].show(config=CFG); notes_fig(v).show(config=CFG)
        if n == 10: display(HTML(CONCL_HTML))


**Live dashboard:** [https://iitm-da25b035.github.io/DA2402-Project-Deploy/](https://iitm-da25b035.github.io/DA2402-Project-Deploy/)